<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/DOC01_geracao_matriz_notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DOC01 — Geração da Matriz de Notebooks v1.5

Notebook documental, somente-leitura sobre as fontes experimentais. Preserva a curadoria textual da Matriz v5 e regenera as partes voláteis a partir do AUD02 v1.2 e do estado físico atual de `04-reports`.

A matriz documental produzida por esta etapa é identificada de forma uniforme como **v7.0** no nome do arquivo, no README, no Dashboard, nas propriedades do workbook e no resumo de execução.

**Arquitetura:** uma única célula de código. Não treina modelos, não refaz splits e não altera notebooks nem artefatos do ramo experimental.


In [ ]:
# ============================================================
# 1. Bootstrap, imports e configuração
# ============================================================

from __future__ import annotations

import copy
import hashlib
import json
import os
import re
import shutil
import sys
import tempfile
import warnings
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

try:
    from openpyxl import load_workbook
    from openpyxl.chart import BarChart, Reference
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.utils import get_column_letter
except ImportError:
    # Portabilidade para Colab limpo. Não afeta o pipeline experimental.
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], check=True)
    from openpyxl import load_workbook
    from openpyxl.chart import BarChart, Reference
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore', category=FutureWarning)

RUN_AT = datetime.now(timezone.utc)
DOC01_VERSION = '1.5'
MATRIX_VERSION = '7.0'
SOURCE_MATRIX_FILENAME = 'Matriz_Notebooks_Dissertacao_v5.xlsx'
OUTPUT_MATRIX_FILENAME = 'Matriz_Notebooks_Dissertacao_v7_0.xlsx'
AUD02_DIRNAME = 'AUD02_evidence_audit'
AUD02_ZIP_PATTERN = 'AUD02_evidence_audit*.zip'
REPORTS_INVENTORY_FILENAME = '04-reports_inventory_full.csv'
AUD02_V12_SENTINELS = (
    'AUD02_summary.json',
    'AUD02_checks.csv',
    'AUD02_pr_auc_lift_fixed_test_by_cell.csv',
    'AUD02_lift_protocol_decision.json',
    'AUD02_feature_contract_audit.json',
    'AUD02_kaggle_comparison.csv',
    'AUD02_artifact_manifest_sha256.csv',
)
MAX_HASH_BYTES = 512 * 1024 * 1024

# ----------------------------
# Ambiente Colab / local
# ----------------------------
IN_COLAB = False
if Path('/content').exists():
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        IN_COLAB = Path('/content/drive/MyDrive').exists()
    except Exception:
        IN_COLAB = False

if IN_COLAB:
    MYDRIVE_ROOT = Path('/content/drive/MyDrive')
    PROJECT_ROOT = MYDRIVE_ROOT / 'Mestrado'
    LOCAL_ROOT = Path('/content')
    OUTPUT_DIR = PROJECT_ROOT / '04-reports' / 'DOC01_matrix_consolidation'
else:
    MYDRIVE_ROOT = Path('/mnt/data')
    PROJECT_ROOT = Path('/mnt/data')
    LOCAL_ROOT = Path('/mnt/data')
    OUTPUT_DIR = Path('/mnt/data') / 'DOC01_matrix_consolidation_v1_4'

REPORTS_ROOT = PROJECT_ROOT / '04-reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Caminhos opcionais. Normalmente devem permanecer None.
# Use-os apenas se houver várias cópias e você quiser fixar explicitamente uma.
SOURCE_MATRIX_OVERRIDE: str | None = None
AUD02_SOURCE_OVERRIDE: str | None = None
REPORTS_INVENTORY_OVERRIDE: str | None = None
ARCHIVE_PATH_OVERRIDES: dict[str, str | None] = {
    'NB99_ABC_Historico.zip': None,
    'NB99_ABC_Saneado.zip': None,
}


def first_existing(candidates: Iterable[Path]) -> Path | None:
    for path in candidates:
        if path and path.exists():
            return path.resolve()
    return None


def safe_mtime(path: Path) -> float:
    try:
        return path.stat().st_mtime
    except Exception:
        return 0.0


def deduplicate_paths(paths: Iterable[Path]) -> list[Path]:
    unique: dict[str, Path] = {}
    for path in paths:
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)
        unique[key] = path
    return list(unique.values())


def recursive_matches(
    roots: Iterable[Path],
    patterns: Iterable[str],
    *,
    files_only: bool = True,
    directories_only: bool = False,
    max_results: int = 500,
) -> list[Path]:
    # Pesquisa recursiva tolerante a cópias com sufixos como "(1)".
    matches: list[Path] = []
    seen: set[str] = set()

    for root in deduplicate_paths(roots):
        if not root.exists():
            continue
        for pattern in patterns:
            try:
                iterator = root.rglob(pattern)
                for path in iterator:
                    if files_only and not path.is_file():
                        continue
                    if directories_only and not path.is_dir():
                        continue
                    try:
                        key = str(path.resolve())
                    except Exception:
                        key = str(path)
                    if key in seen:
                        continue
                    seen.add(key)
                    matches.append(path)
                    if len(matches) >= max_results:
                        return matches
            except (OSError, PermissionError):
                continue

    return matches


def choose_preferred(
    matches: Iterable[Path],
    *,
    exact_name: str | None = None,
) -> Path | None:
    candidates = deduplicate_paths(matches)
    if not candidates:
        return None

    exact = (
        [p for p in candidates if p.name == exact_name]
        if exact_name
        else []
    )
    pool = exact or candidates
    return max(pool, key=safe_mtime).resolve()


def directory_has_aud02_v12(path: Path) -> bool:
    return (
        path.is_dir()
        and all((path / name).is_file() for name in AUD02_V12_SENTINELS)
    )


def zip_has_aud02_v12(path: Path) -> bool:
    if not path.is_file() or path.suffix.lower() != '.zip':
        return False
    try:
        with zipfile.ZipFile(path) as zf:
            basenames = {Path(name).name for name in zf.namelist()}
        return set(AUD02_V12_SENTINELS).issubset(basenames)
    except (OSError, zipfile.BadZipFile):
        return False


# Em Colab, /content é verificado diretamente; o Drive é pesquisado
# recursivamente em MyDrive e, quando montado, em Shared drives.
DIRECT_SEARCH_DIRS = deduplicate_paths([
    LOCAL_ROOT,
    PROJECT_ROOT,
    PROJECT_ROOT / '04-reports',
    MYDRIVE_ROOT,
])

RECURSIVE_SEARCH_ROOTS = deduplicate_paths([
    MYDRIVE_ROOT,
    Path('/content/drive/Shareddrives') if IN_COLAB else LOCAL_ROOT,
])

# ----------------------------
# Localização robusta da Matriz v5
# ----------------------------
matrix_override = (
    Path(SOURCE_MATRIX_OVERRIDE).expanduser()
    if SOURCE_MATRIX_OVERRIDE
    else None
)

matrix_direct = [
    directory / SOURCE_MATRIX_FILENAME
    for directory in DIRECT_SEARCH_DIRS
]

matrix_matches: list[Path] = []
if matrix_override is not None:
    matrix_matches.append(matrix_override)
matrix_matches.extend(
    path for path in matrix_direct if path.exists()
)
matrix_matches.extend(recursive_matches(
    RECURSIVE_SEARCH_ROOTS,
    [
        SOURCE_MATRIX_FILENAME,
        'Matriz_Notebooks_Dissertacao_v5*.xlsx',
        '*Matriz*Notebooks*Dissertacao*v5*.xlsx',
    ],
    files_only=True,
))

SOURCE_MATRIX_PATH = choose_preferred(
    matrix_matches,
    exact_name=SOURCE_MATRIX_FILENAME,
)

if SOURCE_MATRIX_PATH is None:
    similar = recursive_matches(
        RECURSIVE_SEARCH_ROOTS,
        ['*Matriz*Notebooks*.xlsx', '*Dissertacao*v5*.xlsx'],
        files_only=True,
        max_results=30,
    )
    similar_text = '\n'.join(
        f'  - {path}' for path in sorted(similar, key=str)[:20]
    ) or '  - nenhuma planilha semelhante localizada'

    roots_text = '\n'.join(
        f'  - {root}' for root in RECURSIVE_SEARCH_ROOTS
    )
    raise FileNotFoundError(
        f'{SOURCE_MATRIX_FILENAME} não encontrado.\n\n'
        'O DOC01 pesquisou recursivamente em:\n'
        f'{roots_text}\n\n'
        'Arquivos semelhantes encontrados:\n'
        f'{similar_text}\n\n'
        'Soluções possíveis:\n'
        '1. mantenha o arquivo em qualquer pasta de MyDrive; ou\n'
        '2. defina SOURCE_MATRIX_OVERRIDE no início da célula com o caminho completo.'
    )

if len(deduplicate_paths(matrix_matches)) > 1:
    print('[DOC01] Mais de uma Matriz v5 encontrada; selecionada a cópia exata/mais recente:')
    for candidate in sorted(
        deduplicate_paths(matrix_matches),
        key=safe_mtime,
        reverse=True,
    )[:10]:
        marker = '  *' if candidate.resolve() == SOURCE_MATRIX_PATH else '   '
        print(marker, candidate)

# ----------------------------
# Localização robusta do AUD02 v1.2
# ----------------------------
audit_override = (
    Path(AUD02_SOURCE_OVERRIDE).expanduser()
    if AUD02_SOURCE_OVERRIDE
    else None
)

AUD02_DIR = None
audit_zip = None

if audit_override is not None:
    if not audit_override.exists():
        raise FileNotFoundError(
            f'AUD02_SOURCE_OVERRIDE não existe: {audit_override}'
        )
    if audit_override.is_dir():
        if not directory_has_aud02_v12(audit_override):
            raise FileNotFoundError(
                'O diretório definido em AUD02_SOURCE_OVERRIDE não contém '
                f'todos os sentinelas da v1.2: {AUD02_V12_SENTINELS}'
            )
        AUD02_DIR = audit_override.resolve()
    elif audit_override.is_file() and audit_override.suffix.lower() == '.zip':
        if not zip_has_aud02_v12(audit_override):
            raise FileNotFoundError(
                'O ZIP definido em AUD02_SOURCE_OVERRIDE não contém '
                f'todos os sentinelas da v1.2: {AUD02_V12_SENTINELS}'
            )
        audit_zip = audit_override.resolve()

if AUD02_DIR is None and audit_zip is None:
    fixed_audit_dirs = [
        PROJECT_ROOT / '04-reports' / AUD02_DIRNAME,
        PROJECT_ROOT / AUD02_DIRNAME,
        LOCAL_ROOT / AUD02_DIRNAME,
    ]
    complete_fixed_dirs = [
        path for path in fixed_audit_dirs
        if directory_has_aud02_v12(path)
    ]
    AUD02_DIR = choose_preferred(complete_fixed_dirs)

if AUD02_DIR is None and audit_zip is None:
    audit_dirs = recursive_matches(
        RECURSIVE_SEARCH_ROOTS,
        [AUD02_DIRNAME],
        files_only=False,
        directories_only=True,
    )
    complete_dirs = [
        path for path in audit_dirs
        if directory_has_aud02_v12(path)
    ]
    AUD02_DIR = choose_preferred(complete_dirs)

if AUD02_DIR is None and audit_zip is None:
    audit_zips = recursive_matches(
        RECURSIVE_SEARCH_ROOTS,
        [AUD02_ZIP_PATTERN, 'AUD02*evidence*audit*.zip'],
        files_only=True,
    )
    for directory in DIRECT_SEARCH_DIRS:
        if not directory.exists():
            continue
        audit_zips.extend(directory.glob(AUD02_ZIP_PATTERN))

    complete_zips = [
        path for path in deduplicate_paths(audit_zips)
        if zip_has_aud02_v12(path)
    ]
    audit_zip = choose_preferred(complete_zips)

if AUD02_DIR is None:
    if audit_zip is None:
        raise FileNotFoundError(
            'Diretório/ZIP do AUD02 v1.2 não encontrado com todos os '
            f'sentinelas exigidos: {AUD02_V12_SENTINELS}. '
            'O DOC01 pesquisou MyDrive, Shared drives e o diretório local. '
            'Defina AUD02_SOURCE_OVERRIDE no início da célula se necessário.'
        )

    extract_root = OUTPUT_DIR / '_AUD02_extracted'
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(audit_zip) as zf:
        zf.extractall(extract_root)

    audit_candidates: list[Path] = []
    for candidate in [extract_root / AUD02_DIRNAME, extract_root]:
        if directory_has_aud02_v12(candidate):
            audit_candidates.append(candidate)
    if not audit_candidates:
        for sentinel_path in extract_root.rglob('AUD02_summary.json'):
            candidate = sentinel_path.parent
            if directory_has_aud02_v12(candidate):
                audit_candidates.append(candidate)
    AUD02_DIR = choose_preferred(audit_candidates)

if AUD02_DIR is None or not directory_has_aud02_v12(AUD02_DIR):
    raise FileNotFoundError(
        'O pacote AUD02 foi localizado, mas não foi possível resolver '
        'um diretório completo da v1.2 após a extração.'
    )


# ----------------------------
# Inventário físico de 04-reports
# ----------------------------
# O DOC01 v1.4 não seleciona automaticamente inventários históricos.
# Salvo override explícito, ele cria um snapshot do 04-reports vigente
# no momento da execução. O SHA-256 dos artefatos efetivamente resolvidos
# é calculado posteriormente, diretamente no arquivo físico.
inventory_override = (
    Path(REPORTS_INVENTORY_OVERRIDE).expanduser()
    if REPORTS_INVENTORY_OVERRIDE
    else None
)


def build_runtime_reports_inventory(
    reports_root: Path,
    output_path: Path,
) -> Path:
    if not reports_root.exists():
        raise FileNotFoundError(
            f'Diretório 04-reports não encontrado: {reports_root}'
        )

    rows: list[dict[str, Any]] = []
    output_resolved = OUTPUT_DIR.resolve()

    for current, dirs, files in os.walk(reports_root):
        current_path = Path(current)

        # Não indexar a própria saída do DOC01; evita autorreferência e
        # garante que o snapshot represente as fontes prévias à consolidação.
        kept_dirs = []
        for dirname in dirs:
            candidate = current_path / dirname
            try:
                if candidate.resolve() == output_resolved:
                    continue
            except Exception:
                pass
            kept_dirs.append(dirname)
        dirs[:] = kept_dirs

        for filename in files:
            path = current_path / filename
            try:
                resolved = path.resolve()
                if output_resolved in resolved.parents:
                    continue
                stat = path.stat()
                relative = path.relative_to(reports_root).as_posix()
            except (OSError, PermissionError, ValueError):
                continue

            rows.append({
                'relative_path': relative,
                'filename': filename,
                'size_bytes': int(stat.st_size),
                'last_write_time': datetime.fromtimestamp(
                    stat.st_mtime, tz=timezone.utc
                ).isoformat(),
                # Hash vazio de propósito: quando o artefato é usado,
                # o DOC01 calcula o SHA-256 diretamente no arquivo físico.
                'sha256': '',
            })

    inventory_df = pd.DataFrame(
        rows,
        columns=[
            'relative_path',
            'filename',
            'size_bytes',
            'last_write_time',
            'sha256',
        ],
    ).sort_values(
        ['relative_path', 'filename'],
        kind='stable',
    ).reset_index(drop=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    inventory_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    if inventory_df.empty:
        raise RuntimeError(
            f'Inventário runtime vazio para {reports_root}.'
        )

    return output_path.resolve()


if inventory_override is not None:
    if not inventory_override.exists():
        raise FileNotFoundError(
            f'REPORTS_INVENTORY_OVERRIDE não existe: {inventory_override}'
        )
    REPORTS_INVENTORY_PATH = inventory_override.resolve()
    REPORTS_INVENTORY_MODE = 'EXPLICIT_OVERRIDE'
else:
    REPORTS_INVENTORY_PATH = build_runtime_reports_inventory(
        REPORTS_ROOT,
        OUTPUT_DIR / '04-reports_inventory_runtime.csv',
    )
    REPORTS_INVENTORY_MODE = 'RUNTIME_CURRENT_04_REPORTS'


OUTPUT_MATRIX_PATH = OUTPUT_DIR / OUTPUT_MATRIX_FILENAME

print('DOC01_VERSION        :', DOC01_VERSION)
print('MATRIX_VERSION       :', MATRIX_VERSION)
print('IN_COLAB             :', IN_COLAB)
print('PROJECT_ROOT         :', PROJECT_ROOT)
print('SOURCE_MATRIX_PATH   :', SOURCE_MATRIX_PATH)
print('SEARCH_ROOTS         :', [str(p) for p in RECURSIVE_SEARCH_ROOTS])
print('AUD02_DIR            :', AUD02_DIR)
print('REPORTS_INVENTORY    :', REPORTS_INVENTORY_PATH)
print('INVENTORY_MODE       :', REPORTS_INVENTORY_MODE)
print('OUTPUT_DIR           :', OUTPUT_DIR)

# ============================================================
# 2. Funções de leitura, normalização e rastreabilidade
# ============================================================

REQUIRED_AUD02_FILES = [
    'AUD02_summary.json',
    'AUD02_checks.csv',
    'AUD02_claims_audit.csv',
    'AUD02_source_inventory.csv',
    'AUD02_source_hashes.csv',
    'AUD02_modelability_summary.csv',
    'AUD02_nominal_winners_by_cell.csv',
    'AUD02_pr_auc_lift_fixed_test_by_cell.csv',
    'AUD02_operational_metrics_by_cell.csv',
    'AUD02_model_governance_by_cell.csv',
    'AUD02_kaggle_comparison.csv',
    'AUD02_lift_protocol_decision.json',
    'AUD02_feature_contract_audit.json',
    'AUD02_artifact_manifest_sha256.csv',
]

missing_aud02 = [name for name in REQUIRED_AUD02_FILES if not (AUD02_DIR / name).exists()]
if missing_aud02:
    raise FileNotFoundError(f'Artefatos obrigatórios do AUD02 ausentes: {missing_aud02}')


def read_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(AUD02_DIR / name)


def read_json(name: str) -> dict[str, Any]:
    with open(AUD02_DIR / name, 'r', encoding='utf-8') as f:
        return json.load(f)


def sha256_file(path: Path, max_bytes: int = MAX_HASH_BYTES) -> tuple[str | None, str]:
    size = path.stat().st_size
    if size > max_bytes:
        return None, 'SKIPPED_LARGE'
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest(), 'COMPUTED'


# Validação fechada do pacote AUD02 antes de qualquer consolidação.
AUD02_MANIFEST = pd.read_csv(
    AUD02_DIR / 'AUD02_artifact_manifest_sha256.csv',
    dtype=object,
)
required_manifest_cols = {'filename', 'size_bytes', 'sha256'}
missing_manifest_cols = sorted(
    required_manifest_cols - set(AUD02_MANIFEST.columns)
)
if missing_manifest_cols:
    raise ValueError(
        'Manifesto do AUD02 sem colunas obrigatórias: '
        f'{missing_manifest_cols}'
    )

manifest_names = set(
    AUD02_MANIFEST['filename'].astype(str).str.strip()
)
required_not_listed = sorted(
    set(REQUIRED_AUD02_FILES)
    - {'AUD02_artifact_manifest_sha256.csv'}
    - manifest_names
)
if required_not_listed:
    raise RuntimeError(
        'Artefatos obrigatórios não registrados no manifesto final do '
        f'AUD02: {required_not_listed}'
    )

aud02_manifest_issues: list[str] = []
for _, manifest_row in AUD02_MANIFEST.iterrows():
    filename = safe_name = str(
        manifest_row.get('filename', '')
    ).strip()
    if not safe_name:
        continue

    artifact_path = AUD02_DIR / safe_name
    if not artifact_path.is_file():
        aud02_manifest_issues.append(
            f'{safe_name}: MISSING'
        )
        continue

    expected_size = pd.to_numeric(
        manifest_row.get('size_bytes'),
        errors='coerce',
    )
    if pd.notna(expected_size) and artifact_path.stat().st_size != int(expected_size):
        aud02_manifest_issues.append(
            f'{safe_name}: SIZE_MISMATCH'
        )

    expected_sha = str(
        manifest_row.get('sha256', '')
    ).strip().lower()
    if expected_sha:
        observed_sha, observed_status = sha256_file(artifact_path)
        if observed_status != 'COMPUTED' or observed_sha != expected_sha:
            aud02_manifest_issues.append(
                f'{safe_name}: SHA256_MISMATCH'
            )

if aud02_manifest_issues:
    raise RuntimeError(
        'Pacote AUD02 inconsistente com o manifesto final: '
        + ' | '.join(aud02_manifest_issues)
    )



def safe_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ''
    return str(value).strip()


def normalize_filename_token(token: str) -> str:
    return token.strip().strip('.,;:()[]{}<>"\'')


FILE_PATTERN = re.compile(
    r'(?P<name>[A-Za-z0-9À-ÿ_()\-.]+?\.(?:csv|json|parquet|png|pdf|txt|md|ipynb|xlsx|zip))',
    flags=re.IGNORECASE,
)


def extract_filenames(value: Any) -> set[str]:
    text = safe_text(value)
    if not text:
        return set()
    return {normalize_filename_token(m.group('name')) for m in FILE_PATTERN.finditer(text)}


def infer_notebook_id(filename: str) -> str:
    upper = filename.upper()
    if upper.startswith('AUD02'):
        return 'AUD02'
    if upper.startswith('DOC01'):
        return 'DOC01'
    m = re.match(r'^(\d{2})(A)?_FULL', upper)
    if m:
        base = f'NB{m.group(1)}'
        return base + ('A_FULL' if m.group(2) else '_FULL')
    m = re.match(r'^(\d{2})_', upper)
    if m:
        return f'NB{m.group(1)}'
    if upper.startswith('99_A') or upper.startswith('99A'):
        return 'NB99_A'
    if upper.startswith('99_B') or upper.startswith('99B'):
        return 'NB99_B'
    if upper.startswith('99_C') or upper.startswith('99C'):
        return 'NB99_C'
    return ''


def remap_project_path(raw_path: Any) -> Path | None:
    text = safe_text(raw_path)
    if not text:
        return None

    normalized = text.replace('\\', '/')
    p = Path(text)
    if p.exists():
        return p.resolve()

    # Caminho relativo ao 04-reports.
    if not re.match(r'^[A-Za-z]:/', normalized) and not normalized.startswith('/'):
        candidate = REPORTS_ROOT.joinpath(*normalized.split('/'))
        if candidate.exists():
            return candidate.resolve()

    marker = '/Mestrado/'
    if marker in normalized:
        suffix = normalized.split(marker, 1)[1]
        candidate = PROJECT_ROOT.joinpath(*suffix.split('/'))
        if candidate.exists():
            return candidate.resolve()
    return None


def normalize_relative_path(value: Any) -> str:
    text = safe_text(value).replace('/', '\\')
    return re.sub(r'\\+', r'\\', text).lstrip('\\')


def load_reports_inventory(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=object)
    required = {'relative_path', 'filename', 'size_bytes', 'last_write_time', 'sha256'}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'Inventário de 04-reports sem colunas obrigatórias: {missing}')
    df = df.copy()
    df['relative_path'] = df['relative_path'].map(normalize_relative_path)
    df['filename'] = df['filename'].map(safe_text)
    df['sha256'] = df['sha256'].map(lambda x: safe_text(x).lower())
    df['size_bytes'] = pd.to_numeric(df['size_bytes'], errors='coerce')
    df['last_write_time'] = df['last_write_time'].map(safe_text)
    df = df[df['filename'] != ''].drop_duplicates(
        subset=['relative_path', 'filename', 'sha256'],
        keep='last',
    ).reset_index(drop=True)
    return df


REPORTS_INVENTORY = load_reports_inventory(REPORTS_INVENTORY_PATH)
INVENTORY_INDEX: dict[str, list[dict[str, Any]]] = defaultdict(list)
for _, row in REPORTS_INVENTORY.iterrows():
    relative = normalize_relative_path(row['relative_path'])
    full_path = REPORTS_ROOT.joinpath(*relative.split('\\'))
    INVENTORY_INDEX[safe_text(row['filename'])].append({
        'filename': safe_text(row['filename']),
        'relative_path': relative,
        'full_path': full_path,
        'size_bytes': int(row['size_bytes']) if pd.notna(row['size_bytes']) else None,
        'last_write_time': safe_text(row['last_write_time']),
        'sha256': safe_text(row['sha256']).lower(),
    })


def make_file_index(roots: list[Path]) -> dict[str, list[Path]]:
    # Fallback para arquivos fora de 04-reports, sobretudo pacotes históricos.
    index: dict[str, list[Path]] = defaultdict(list)
    allowed = {'.csv', '.json', '.parquet', '.png', '.pdf', '.txt', '.md', '.ipynb', '.xlsx', '.zip'}
    seen_roots = set()
    for root in roots:
        if not root.exists():
            continue
        try:
            resolved = str(root.resolve())
        except Exception:
            resolved = str(root)
        if resolved in seen_roots:
            continue
        seen_roots.add(resolved)
        for current, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if d not in {'_AUD02_extracted', '__pycache__', 'DOC01_matrix_consolidation'}
            ]
            for name in files:
                path = Path(current) / name
                if path.suffix.lower() in allowed:
                    index[name].append(path)
    return index


def canonical_candidate_score(record: dict[str, Any], notebook_ids: list[str]) -> int:
    relative = safe_text(record.get('relative_path')).replace('\\', '/').lower()
    filename = safe_text(record.get('filename')).lower()
    parts = [part for part in relative.split('/') if part]
    score = 0

    # Penalizações fortes para cópias temporárias e históricas.
    if any(part.startswith('_') for part in parts[:-1]):
        score -= 100
    if any(token in relative for token in ['backup', 'historico', 'historical', 'previous', '/old/', '/temp/']):
        score -= 80
    if any(part.startswith('dry_run') for part in parts):
        score -= 40
    if any(part.startswith('execute_') for part in parts):
        score -= 30

    # Agregados são preferidos para summaries, allcells e visões by_cell.
    if '/aggregate/' in relative:
        if any(token in filename for token in [
            'allcells', 'by_cell', 'summary', 'manifest', 'aggregate',
            'decision', 'inventory', 'limitations', 'registry', 'checks',
        ]):
            score += 25
        else:
            score += 5

    # Afinidade com o notebook relacionado.
    for notebook_id in notebook_ids:
        token = safe_text(notebook_id).lower().replace('nb', '')
        if token and token in relative:
            score += 8

    # Menor profundidade como desempate documental.
    score -= max(len(parts) - 1, 0) * 2
    return score


def choose_inventory_candidate(
    filename: str,
    candidates: list[dict[str, Any]],
    notebook_ids: list[str],
    relative_hints: Iterable[str] = (),
) -> dict[str, Any]:
    result = {
        'record': None,
        'resolution_status': 'NOT_FOUND',
        'resolution_confidence': 'LOW',
        'candidate_count': len(candidates),
        'distinct_hash_count': len({safe_text(c.get('sha256')) for c in candidates if safe_text(c.get('sha256'))}),
        'candidate_paths': ' | '.join(sorted(safe_text(c.get('relative_path')) for c in candidates)),
    }
    if not candidates:
        return result

    normalized_hints = {
        normalize_relative_path(hint)
        for hint in relative_hints
        if safe_text(hint)
    }
    for candidate in candidates:
        if normalize_relative_path(candidate.get('relative_path')) in normalized_hints:
            result.update({
                'record': candidate,
                'resolution_status': 'INVENTORY_PATH_HINT_MATCH',
                'resolution_confidence': 'HIGH',
            })
            return result

    if len(candidates) == 1:
        result.update({
            'record': candidates[0],
            'resolution_status': 'INVENTORY_UNIQUE',
            'resolution_confidence': 'HIGH',
        })
        return result

    scored = []
    for candidate in candidates:
        scored.append((canonical_candidate_score(candidate, notebook_ids), candidate))
    max_score = max(score for score, _ in scored)
    top = [candidate for score, candidate in scored if score == max_score]

    if len(top) == 1:
        result.update({
            'record': top[0],
            'resolution_status': 'CANONICAL_RULE_SELECTED',
            'resolution_confidence': 'HIGH',
        })
        return result

    top_hashes = {safe_text(candidate.get('sha256')) for candidate in top if safe_text(candidate.get('sha256'))}
    if len(top_hashes) == 1:
        chosen = sorted(
            top,
            key=lambda c: (
                normalize_relative_path(c.get('relative_path')).count('\\'),
                len(normalize_relative_path(c.get('relative_path'))),
                normalize_relative_path(c.get('relative_path')).lower(),
            ),
        )[0]
        result.update({
            'record': chosen,
            'resolution_status': 'CANONICAL_HASH_IDENTICAL',
            'resolution_confidence': 'HIGH',
        })
        return result

    chosen = sorted(
        top,
        key=lambda c: (
            normalize_relative_path(c.get('relative_path')).count('\\'),
            len(normalize_relative_path(c.get('relative_path'))),
            normalize_relative_path(c.get('relative_path')).lower(),
        ),
    )[0]
    result.update({
        'record': chosen,
        'resolution_status': 'AMBIGUOUS_DIFFERENT_CONTENT',
        'resolution_confidence': 'LOW',
    })
    return result


def dataframe_from_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
    return pd.read_excel(path, sheet_name=sheet_name, dtype=object, engine='openpyxl')


def load_static_sheets(path: Path) -> dict[str, pd.DataFrame]:
    required = [
        'Indice_Notebooks',
        'Metodo_Ciencia',
        'Escrita_Dissertacao',
        'Defesa_Validade',
        'Acoes_Revisao',
    ]
    return {name: dataframe_from_sheet(path, name) for name in required}


# AUD02 atual
AUD02_SUMMARY = read_json('AUD02_summary.json')
AUD02_CHECKS = read_csv('AUD02_checks.csv')
AUD02_CLAIMS = read_csv('AUD02_claims_audit.csv')
AUD02_SOURCES = read_csv('AUD02_source_inventory.csv')
AUD02_SOURCE_HASHES = read_csv('AUD02_source_hashes.csv')
MODELABILITY = read_csv('AUD02_modelability_summary.csv')
WINNERS = read_csv('AUD02_nominal_winners_by_cell.csv')
LIFT_FIXED = read_csv('AUD02_pr_auc_lift_fixed_test_by_cell.csv')
OPERATIONAL = read_csv('AUD02_operational_metrics_by_cell.csv')
MODEL_GOV = read_csv('AUD02_model_governance_by_cell.csv')
KAGGLE_COMPARISON = read_csv('AUD02_kaggle_comparison.csv')
LIFT_DECISION = read_json('AUD02_lift_protocol_decision.json')


def require_columns(
    df: pd.DataFrame,
    label: str,
    required_columns: set[str],
) -> None:
    missing = sorted(required_columns - set(df.columns))
    if missing:
        raise ValueError(
            f'{label} sem colunas obrigatórias para o DOC01: {missing}'
        )


require_columns(
    AUD02_CHECKS,
    'AUD02_checks.csv',
    {'check_id', 'check_class', 'status'},
)
require_columns(
    AUD02_CLAIMS,
    'AUD02_claims_audit.csv',
    {
        'claim_id', 'claim_text', 'module', 'status', 'result',
        'evidence_files', 'evidence_fields', 'derivation_rule',
        'source_hashes_sha256', 'notes',
    },
)
require_columns(
    AUD02_SOURCES,
    'AUD02_source_inventory.csv',
    {'source_key', 'path', 'filename', 'manifest_status', 'manifest_sha256'},
)
require_columns(
    AUD02_SOURCE_HASHES,
    'AUD02_source_hashes.csv',
    {'source_key', 'path', 'sha256'},
)
require_columns(
    MODELABILITY,
    'AUD02_modelability_summary.csv',
    {
        'cell_id', 'is_modelable', 'cell_modelability_status',
        'n_scenarios', 'n_modelable_scenarios', 'min_pos_test',
        'n_total_episodes', 'n_evaluable_episodes_tau_reference',
        'scenario_label',
    },
)
require_columns(
    WINNERS,
    'AUD02_nominal_winners_by_cell.csv',
    {
        'cell_id', 'scenario_label', 'feature_set', 'model',
        'n_folds_valid', 'f1_tscv_mean', 'f1_tscv_std',
        'roc_auc_tscv_mean', 'roc_auc_tscv_std',
        'average_precision_tscv_mean', 'average_precision_tscv_std',
        'n_positivos_test',
    },
)
require_columns(
    LIFT_FIXED,
    'AUD02_pr_auc_lift_fixed_test_by_cell.csv',
    {
        'cell_id', 'average_precision_score_calibrated',
        'prevalence_fixed_test', 'lift_pr_fixed_test',
        'protocol_alignment',
    },
)
require_columns(
    OPERATIONAL,
    'AUD02_operational_metrics_by_cell.csv',
    {
        'cell_id', 'tau_reference', 'f1_tau_reference',
        'false_alerts_per_day_tau_reference',
        'episode_anticipation_rate_tau_reference',
        'lead_time_minutes_median_tau_reference',
        'n_anticipated_episodes_tau_reference',
    },
)
require_columns(
    MODEL_GOV,
    'AUD02_model_governance_by_cell.csv',
    {
        'cell_id', 'baseline_scenario', 'lstm_f1', 'delta_f1',
        'delta_f1_se_ratio', 'nominal_delta_gate_pass',
        'robustness_gate_pass', 'promoted_as_primary_in_nb14',
        'evaluated_as_secondary_in_nb14', 'final_lstm_position',
        'score_source_coherence',
    },
)
require_columns(
    KAGGLE_COMPARISON,
    'AUD02_kaggle_comparison.csv',
    {
        'cell_id', 'scenario_label', 'benchmark_roc_auc',
        'delta_roc_auc', 'supera_referencia', 'is_modelable',
    },
)

required_summary_keys = {
    'overall_status',
    'snapshot_id',
    'nb16_consumed',
    'counts',
    'headline_results',
}
missing_summary_keys = sorted(
    required_summary_keys - set(AUD02_SUMMARY)
)
if missing_summary_keys:
    raise ValueError(
        f'AUD02_summary.json sem chaves obrigatórias: {missing_summary_keys}'
    )

headline_required = {
    'observed_cells', 'modelable_cells', 'roc_auc_min', 'roc_auc_max',
    'n_above_reference', 'anticipation_median', 'lift_min', 'lift_max',
    'lstm_governance',
}
missing_headline = sorted(
    headline_required - set(AUD02_SUMMARY['headline_results'])
)
if missing_headline:
    raise ValueError(
        'AUD02_summary.json/headline_results sem chaves obrigatórias: '
        f'{missing_headline}'
    )

if AUD02_SUMMARY.get('nb16_consumed') is not False:
    raise RuntimeError(
        'AUD02_summary.json indica consumo do NB16_FULL; '
        'isso viola a política vigente de não circularidade.'
    )


STATIC_SHEETS = load_static_sheets(SOURCE_MATRIX_PATH)

if str(AUD02_SUMMARY.get('overall_status', '')).startswith('FAIL'):
    raise RuntimeError(f"AUD02 contém falha material: {AUD02_SUMMARY.get('overall_status')}")

print('AUD02 checks:', AUD02_CHECKS['status'].value_counts(dropna=False).to_dict())
print('Static sheets loaded:', {k: len(v) for k, v in STATIC_SHEETS.items()})

# ============================================================
# 3. Curadoria durável da v5 e correções críticas v7
# ============================================================

CURATION_DATE = RUN_AT.date().isoformat()

# Atualizações deliberadamente pequenas: apenas formulações que ficaram
# objetivamente desatualizadas após o AUD02 v1.2 e o fechamento do ramo FULL.
OVERRIDES: dict[str, dict[str, dict[str, Any]]] = {
    'Indice_Notebooks': {
        'NB11_FULL': {
            'observacao_indice': (
                'Seleção soberana por célula sob TSCV: HistGradientBoosting em b, c, e e g; '
                'Regressão Logística em a, f e h. Sete células modeláveis; d é não modelável. '
                'ROC-AUC soberano nas modeláveis: 0,637–0,813.'
            ),
            'status_acao': 'concluída',
        },
        'NB13_FULL': {
            'observacao_indice': (
                'LSTM complementar por célula. apenas b ultrapassa o piso nominal ΔF1≥0,03; '
                'nenhuma célula supera ΔF1/SE≥2 e nenhuma LSTM é promovida.'
            ),
        },
        'NB13a_FULL': {
            'observacao_indice': (
                'Busca sistemática em 1.491 avaliações. Confirma a não promoção da LSTM; '
                'a célula d/P2 é bloqueada pelo gate de modelabilidade, não por erro de execução.'
            ),
            'status_acao': 'concluída',
        },
        'NB14_FULL': {
            'observacao_indice': (
                'Avaliação operacional por célula no teste fixo herdado: τ, custo, falsos alertas, '
                'antecipação e lead time. Fonte primária NB11_FULL em todas as células.'
            ),
            'status_acao': 'concluída',
        },
        'NB16_FULL': {
            'observacao_indice': (
                'Fechamento do ramo experimental FULL: 1.150/1.150 artefatos e 19/19 verificações OK. '
                'Permanece inalterado; não consome o AUD02.'
            ),
            'status_acao': 'concluída',
        },
    },
    'Metodo_Ciencia': {
        'NB11_FULL': {
            'objetivo_notebook': 'Treinar e selecionar o resultado soberano por célula nos cenários promovidos pelo NB10_FULL.',
            'logica_funcionamento': (
                'Avalia modelos tabulares por célula sob TimeSeriesSplit; seleciona por F1 médio, '
                'com desempate por recall e PR-AUC. O cenário soberano pode variar entre células.'
            ),
            'artefatos_citaveis': (
                '11_FULL_metrics_summary_by_cell.csv; 11_FULL_metrics_tscv_by_cell.csv; '
                '11_FULL_modelability_cell_overview.csv; 11_FULL_scores_calibrated_by_cell.parquet.'
            ),
            'achados_cientificos': (
                'Sete células são modeláveis. ROC-AUC TSCV soberano entre 0,636798 (f) e 0,812583 (g). '
                'Seis das sete células modeláveis superam descritivamente o valor de referência 0,647358; '
                'f fica abaixo e d não integra a comparação.'
            ),
            'lacunas_cientificas': (
                'A célula d é não modelável em todos os cenários avaliados; possui suporte positivo insuficiente '
                'no teste e apenas um episódio avaliável no cenário operacional selecionado.'
            ),
            'variacao_entre_celulas': 'Alta e central: cenário, modelo, discriminação e utilidade operacional variam por célula.',
            'conceitos_estatisticos': 'TimeSeriesSplit; seleção soberana por F1; desempate por recall e PR-AUC; gate de modelabilidade.',
            'importancia_conceitos_estatisticos': (
                'Fornece a evidência principal de discriminação do experimento FULL sem colapsar as células em uma média única.'
            ),
            'justificativa_confianca': 'Resultados soberanos e modelabilidade reproduzidos pelo AUD02 v1.2, com hashes e checks PASS.',
        },
        'NB13_FULL': {
            'achados_cientificos': (
                'apenas b supera o piso nominal ΔF1≥0,03. Nenhuma célula supera o gate descritivo '
                'ΔF1/SE≥2 e nenhuma LSTM é promovida como fonte primária.'
            ),
            'achados_metodologicos': (
                'A razão ΔF1/SE é heurística descritiva não pareada, não teste de significância. '
                'Os escores LSTM seguem apenas como sensibilidades secundárias no NB14_FULL.'
            ),
            'conceitos_estatisticos': 'Gate nominal ΔF1≥0,03; gate descritivo de robustez ΔF1/SE≥2; comparação não pareada.',
            'justificativa_confianca': 'Governança cruzada NB13/NB13a/NB14 confirmada no AUD02 v1.2.',
        },
        'NB13a_FULL': {
            'achados_metodologicos': (
                'O tuning ampliado confirma a recomendação soberana de manter NB11_FULL como fonte primária. '
                'NB13_FULL e NB13a_FULL são avaliados apenas como sensibilidades secundárias.'
            ),
            'lacunas_cientificas': (
                'O issue d/P2 corresponde a suporte positivo insuficiente (n_positivos_test=12<30), '
                'isto é, aplicação do gate de modelabilidade.'
            ),
            'justificativa_confianca': '1.491 avaliações registradas; decisão final reconciliada pelo AUD02 v1.2.',
        },
        'NB14_FULL': {
            'logica_funcionamento': (
                'Consome a fonte primária NB11_FULL e avalia métricas operacionais no teste fixo herdado. '
                'LSTMs aparecem somente em linhas secundárias de sensibilidade.'
            ),
            'artefatos_citaveis': (
                '14_FULL_scenario_summary_by_cell.csv; 14_FULL_optimal_tau_by_cost_allcells.csv; '
                '14_FULL_threshold_metrics_by_tau_allcells.csv.'
            ),
            'achados_cientificos': (
                'Não existe τ universal. No limiar de referência τ=0,5, a mediana da antecipação episódica '
                'entre as sete células modeláveis é 33,33%, com forte heterogeneidade.'
            ),
            'justificativa_confianca': 'Métricas operacionais e antecipação reproduzidas pelo AUD02 v1.2.',
        },
        'NB16_FULL': {
            'objetivo_notebook': 'Consolidar e selar documentalmente o ramo experimental FULL.',
            'logica_funcionamento': 'Inventaria 1.150 artefatos, verifica paridade do manifesto, executa 19 checks e registra decisões e limitações com evidência.',
            'saidas_principais': '16_FULL_artifact_inventory.csv; 16_FULL_artifact_manifest_sha256.csv; 16_FULL_decision_log.csv; 16_FULL_limitations.csv; 16_FULL_dissertation_figures_registry.csv.',
            'artefatos_citaveis': '16_FULL_decision_log.csv; 16_FULL_limitations.csv; 16_FULL_dissertation_figures_registry.csv; 16_FULL_artifact_manifest_sha256.csv.',
            'achados_cientificos': 'Não gera resultado preditivo novo; sela inventário, decisões, limitações e consistência do ramo FULL.',
            'achados_metodologicos': 'Fechamento final com 1.150/1.150 artefatos e 19/19 verificações OK.',
            'lacunas_cientificas': 'Limitações permanecem documentadas; o AUD02 é auditoria transversal externa ao ramo.',
            'justificativa_confianca': 'Artefatos e verificações finais validados após a última reexecução.',
        },
    },
    'Escrita_Dissertacao': {
        'NB11_FULL': {
            'aproveitamento_capitulo_4': 'Resultados soberanos das sete células modeláveis; célula d apresentada separadamente como não modelável.',
            'aproveitamento_capitulo_5': 'Heterogeneidade de cenário, modelo, discriminação e suporte entre células.',
            'tabelas_candidatas': 'Tabela soberana por célula: modelabilidade, cenário, modelo, F1, ROC-AUC, PR-AUC, lift alinhado e métricas operacionais.',
            'problemas_md_final': 'Nenhum problema narrativo pendente após o saneamento de julho/2026.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Usar somente resultados soberanos; seis das sete células modeláveis superam descritivamente a referência; d não entra na comparação.',
        },
        'NB12_FULL': {
            'problemas_md_final': 'Narrativa de reenquadramento corrigida na reexecução de 04/07/2026.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Usar como sensibilidade e discussão da tensão entre desempenho local e governança.',
        },
        'NB13_FULL': {
            'aproveitamento_capitulo_4': 'Resultado complementar: apenas b passa pelo piso nominal; nenhuma promoção.',
            'aproveitamento_capitulo_5': 'Justificativa da não promoção da LSTM com dois gates distintos.',
            'problemas_md_final': 'Nenhum problema factual; evitar linguagem inferencial para ΔF1/SE.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Nenhuma LSTM é fonte primária; NB13/NB13a aparecem apenas como sensibilidades secundárias no NB14_FULL.',
        },
        'NB13a_FULL': {
            'aproveitamento_capitulo_4': 'Síntese da busca de 1.491 avaliações e decisão final de não promoção.',
            'aproveitamento_capitulo_5': 'Evidência contra a hipótese de tuning insuficiente.',
            'problemas_md_final': 'O issue d/P2 é gate de modelabilidade, não erro.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Relatar busca sistemática em espaço finito e limitar a conclusão ao espaço testado.',
        },
        'NB14_FULL': {
            'aproveitamento_capitulo_4': 'Métricas operacionais no teste fixo herdado por célula; antecipação mediana 33,33% nas modeláveis.',
            'aproveitamento_capitulo_5': 'Não existe política universal de τ; utilidade operacional é heterogênea.',
            'problemas_md_final': 'Narrativa de reenquadramento corrigida; nenhuma pendência metodológica do notebook.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Distinguir claramente TSCV de seleção e teste fixo de avaliação operacional.',
        },
        'NB16_FULL': {
            'aproveitamento_capitulo_3': 'Governança, inventário, manifesto e rastreabilidade do ramo experimental.',
            'aproveitamento_capitulo_4': 'Não gera métrica preditiva nova.',
            'figuras_candidatas': 'Registro final: 4 figuras agregadas no corpo; figuras por célula em apêndice, com temporal_zoom no corpo apenas por decisão editorial do orientador.',
            'aproveitamento_capitulo_5': 'Decisões, limitações e reprodutibilidade.',
            'problemas_md_final': 'DF08/LF07 e números finais corrigidos; ramo selado.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Usar como fechamento do ramo FULL. AUD02 é auditoria transversal posterior e independente.',
        },
    },
    'Defesa_Validade': {
        'NB11_FULL': {
            'respostas_banca': (
                'Os resultados são apresentados por célula e por resultado soberano. Sete células são modeláveis; '
                'a célula d é explicitamente excluída das faixas. Seis das sete modeláveis superam descritivamente '
                'o valor de referência do protótipo Kaggle refinado.'
            ),
            'como_mitigar_na_dissertacao': 'Reportar tabela completa por célula, protocolo TSCV e regra de seleção soberana.',
        },
        'NB13_FULL': {
            'respostas_banca': (
                'A LSTM foi testada e ajustada. apenas b passa pelo piso nominal; nenhuma célula passa '
                'pelo gate ΔF1/SE≥2. Nenhuma LSTM é promovida; o custo computacional é argumento acessório.'
            ),
            'como_mitigar_na_dissertacao': 'Descrever os dois gates e evitar linguagem de significância estatística.',
        },
        'NB13a_FULL': {
            'respostas_banca': (
                'Foram realizadas 1.491 avaliações. O tuning não alterou a decisão soberana. '
                'O issue d/P2 é bloqueio de modelabilidade por suporte insuficiente.'
            ),
            'lacuna_para_orientador': 'Nenhuma pendência experimental; apenas transposição para o texto.',
        },
        'NB14_FULL': {
            'respostas_banca': (
                'O framework converte escores em decisões por célula, no teste fixo herdado, com τ, custo, '
                'falsos alertas, antecipação e lead time. Não há política única universal.'
            ),
            'como_mitigar_na_dissertacao': 'Fixar explicitamente o protocolo do teste fixo para métricas operacionais.',
        },
        'NB16_FULL': {
            'respostas_banca': (
                'O ramo FULL foi fechado com 1.150/1.150 artefatos e 19/19 verificações OK. '
                'O AUD02 não altera nem reabre o NB16_FULL.'
            ),
            'ameacas_validade': 'Risco residual apenas de interpretação documental, mitigado pela separação entre ramo experimental e auditoria transversal.',
            'como_mitigar_na_dissertacao': 'Referenciar o NB16_FULL como selo do ramo e o AUD02 como validação transversal posterior.',
            'lacuna_para_orientador': 'Nenhuma decisão de reexecução pendente.',
        },
    },
}


def apply_overrides(df: pd.DataFrame, sheet_name: str) -> pd.DataFrame:
    out = df.copy()
    if 'notebook_id' not in out.columns:
        return out
    if 'curadoria_v7' not in out.columns:
        out['curadoria_v7'] = 'MIGRADO_V5'
    if 'fonte_atualizacao_v7' not in out.columns:
        out['fonte_atualizacao_v7'] = 'Matriz v5'
    for notebook_id, changes in OVERRIDES.get(sheet_name, {}).items():
        mask = out['notebook_id'].astype(str) == notebook_id
        if not mask.any():
            continue
        for column, value in changes.items():
            if column in out.columns:
                out.loc[mask, column] = value
        out.loc[mask, 'curadoria_v7'] = 'ATUALIZADO_DOC01'
        out.loc[mask, 'fonte_atualizacao_v7'] = 'AUD02 v1.2 / fechamento FULL'
    return out


for sheet_name in list(STATIC_SHEETS):
    STATIC_SHEETS[sheet_name] = apply_overrides(STATIC_SHEETS[sheet_name], sheet_name)

# Normalização terminológica segura em texto curado. Nomes físicos de arquivos
# em inglês (por exemplo, *_canonical.*) não são alterados.
TEXT_REPLACEMENTS = [
    (r'\bcanônic[oa]s?\b', 'de referência'),
    (r'6 das 8 células', 'seis das sete células modeláveis'),
    (r'6 de 8 células', 'seis das sete células modeláveis'),
    (r'7 das 8 células', 'seis das sete células modeláveis'),
]

def normalize_curated_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for column in out.columns:
        if out[column].dtype == object:
            for pattern, replacement in TEXT_REPLACEMENTS:
                out[column] = out[column].map(
                    lambda value: re.sub(pattern, replacement, value, flags=re.IGNORECASE)
                    if isinstance(value, str) else value
                )
    return out

for sheet_name in list(STATIC_SHEETS):
    STATIC_SHEETS[sheet_name] = normalize_curated_text(STATIC_SHEETS[sheet_name])


# Inclusão dos notebooks documentais sem inseri-los no ramo experimental.
NEW_INDEX_ROWS = [
    {
        'pipeline': 'Documentação/Apoio',
        'fase_metodologica': 'governança',
        'notebook_id': 'AUD02',
        'notebook_nome': 'Auditoria Integrada de Evidências',
        'versao_data': '2026-07-26 — v1.2',
        'status_execucao': 'executado',
        'resultado_aproveitavel': 'central para rastreabilidade',
        'grau_confianca': 'alto',
        'prioridade_revisao': 'baixa',
        'status_acao': 'concluída',
        'observacao_indice': 'Auditoria transversal somente-leitura; 19 checks (18 PASS + 1 REVIEW R10 não bloqueante); não consome NB16_FULL.',
        'curadoria_v7': 'NOVO_DOC01',
        'fonte_atualizacao_v7': 'AUD02 v1.2',
    },
    {
        'pipeline': 'Documentação/Apoio',
        'fase_metodologica': 'síntese',
        'notebook_id': 'DOC01',
        'notebook_nome': 'Geração da Matriz de Notebooks',
        'versao_data': f'{CURATION_DATE} — v{DOC01_VERSION}',
        'status_execucao': 'executado ao gerar a matriz',
        'resultado_aproveitavel': 'apoio',
        'grau_confianca': 'alto',
        'prioridade_revisao': 'baixa',
        'status_acao': 'concluída',
        'observacao_indice': 'Consolida artefatos, campos, protocolos, hashes e valores atuais; não produz evidência experimental.',
        'curadoria_v7': 'NOVO_DOC01',
        'fonte_atualizacao_v7': 'DOC01',
    },
]

index_df = STATIC_SHEETS['Indice_Notebooks']
for row in NEW_INDEX_ROWS:
    if row['notebook_id'] not in set(index_df['notebook_id'].astype(str)):
        for col in index_df.columns:
            row.setdefault(col, '')
        index_df = pd.concat([index_df, pd.DataFrame([row])[index_df.columns]], ignore_index=True)
STATIC_SHEETS['Indice_Notebooks'] = index_df

# Textos documentais concisos para AUD02/DOC01 nas demais abas.
DOC_ROWS = {
    'Metodo_Ciencia': [
        {
            'notebook_id': 'AUD02',
            'objetivo_notebook': 'Auditar claims centrais da escrita com regras, fontes, campos e hashes.',
            'logica_funcionamento': 'Lê artefatos congelados, reproduz derivações e executa checks estruturais, de regressão e rastreabilidade.',
            'entradas_principais': 'Protótipo Kaggle refinado e NB04_FULL–NB15_FULL.',
            'saidas_principais': 'Claims, checks, inventário de fontes, governança por célula, lift alinhado e manifesto.',
            'artefatos_citaveis': 'AUD02_claims_audit.csv; AUD02_checks.csv; AUD02_summary.json; AUD02_source_inventory.csv.',
            'achados_cientificos': 'Não produz achado novo; confirma e qualifica evidências existentes.',
            'achados_metodologicos': 'Separa fonte primária, derivação auditada e cautela metodológica.',
            'lacunas_cientificas': 'Escopo limitado às claims definidas; não substitui inventário geral do pipeline.',
            'variacao_entre_celulas': 'Reproduzida quando relevante.',
            'modelos_algoritmos': 'Nenhum treinamento.',
            'conceitos_estatisticos': 'Rastreabilidade, protocolo, reprodução determinística e checks.',
            'importancia_conceitos_estatisticos': 'Evita que números e decisões entrem na dissertação sem origem verificável.',
            'justificativa_confianca': '19 checks: 18 PASS + 1 REVIEW R10 não bloqueante; manifesto SHA-256 íntegro.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'AUD02 v1.2',
        },
        {
            'notebook_id': 'DOC01',
            'objetivo_notebook': 'Gerar a matriz de consulta rápida sem retipar números.',
            'logica_funcionamento': 'Preserva curadoria textual da v5 e regenera abas voláteis a partir dos artefatos.',
            'entradas_principais': 'Matriz v5; diretório de artefatos do AUD02; artefatos oficiais indexados.',
            'saidas_principais': 'Matriz v7; registros CSV; validação; resumo; manifesto.',
            'artefatos_citaveis': 'Não é fonte científica; é instrumento de navegação e conferência.',
            'achados_cientificos': 'Nenhum.',
            'achados_metodologicos': 'Separa conteúdo curado de conteúdo gerado automaticamente.',
            'lacunas_cientificas': 'Artefatos não cobertos pelo AUD02 são indexados, não auditados.',
            'variacao_entre_celulas': 'Consolidada na aba FULL_por_Celula.',
            'modelos_algoritmos': 'Nenhum treinamento.',
            'conceitos_estatisticos': 'Nenhum cálculo experimental novo.',
            'importancia_conceitos_estatisticos': 'Reduz manutenção paralela e divergências documentais.',
            'justificativa_confianca': 'Falha fechada para entradas obrigatórias e registra nível de validação por linha.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'DOC01',
        },
    ],
    'Escrita_Dissertacao': [
        {
            'notebook_id': 'AUD02',
            'capitulos_relacionados': 'Metodologia; Resultados; Discussão; Apêndice de reprodutibilidade',
            'secoes_relacionadas': 'Rastreabilidade das evidências',
            'aproveitamento_capitulo_1': 'Não citar como fonte experimental.',
            'aproveitamento_capitulo_3': 'Descrição breve da disciplina de rastreabilidade.',
            'aproveitamento_capitulo_4': 'Validação transversal das afirmações numéricas.',
            'aproveitamento_capitulo_5': 'Qualificações e limitações.',
            'aproveitamento_conclusao': 'Não necessário.',
            'figuras_candidatas': 'Nenhuma.',
            'tabelas_candidatas': 'Claims e protocolos, preferencialmente em apêndice.',
            'texto_md_aproveitavel': 'parcial',
            'qualidade_md_inicial': 'excelente',
            'problemas_md_inicial': 'Nenhum.',
            'qualidade_md_final': 'excelente',
            'problemas_md_final': 'Nenhum.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Usar seletivamente; a fonte experimental continua sendo o artefato original.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'AUD02 v1.2',
        },
        {
            'notebook_id': 'DOC01',
            'capitulos_relacionados': 'Documentação interna',
            'secoes_relacionadas': 'Não citar no corpo como método experimental',
            'aproveitamento_capitulo_1': 'Não aplicável.',
            'aproveitamento_capitulo_3': 'No máximo nota de organização/reprodutibilidade.',
            'aproveitamento_capitulo_4': 'Não aplicável.',
            'aproveitamento_capitulo_5': 'Não aplicável.',
            'aproveitamento_conclusao': 'Não aplicável.',
            'figuras_candidatas': 'Nenhuma.',
            'tabelas_candidatas': 'A matriz é instrumento de trabalho, não tabela científica.',
            'texto_md_aproveitavel': 'não',
            'qualidade_md_inicial': 'boa',
            'problemas_md_inicial': 'Não aplicável.',
            'qualidade_md_final': 'boa',
            'problemas_md_final': 'Não aplicável.',
            'termos_a_atualizar': 'nenhum',
            'observacoes_escrita': 'Instrumento interno de navegação e conferência.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'DOC01',
        },
    ],
    'Defesa_Validade': [
        {
            'notebook_id': 'AUD02',
            'questionamentos_banca': 'A rastreabilidade depende apenas de planilhas manuais?',
            'respostas_banca': 'Não. As claims centrais são reproduzidas por auditoria somente-leitura com fontes, campos, regras e hashes.',
            'ameacas_validade': 'Confundir aprovação dos checks com ausência de limitações metodológicas.',
            'como_mitigar_na_dissertacao': 'Manter cautelas em prosa e citar o artefato experimental original.',
            'lacuna_para_orientador': 'Nenhuma.',
            'relacao_artigo_sbpo': 'não relacionado',
            'relacao_experimento_full': 'audita',
            'observacoes_defesa': 'AUD02 não reabre o ramo experimental.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'AUD02 v1.2',
        },
        {
            'notebook_id': 'DOC01',
            'questionamentos_banca': 'A matriz é uma segunda fonte de verdade?',
            'respostas_banca': 'Não. Valores são lidos dos artefatos; a matriz registra caminho, campo, seletor, protocolo e nível de validação.',
            'ameacas_validade': 'Uso de linha apenas indexada como se fosse auditada.',
            'como_mitigar_na_dissertacao': 'Observar a coluna validation_level e retornar à fonte primária.',
            'lacuna_para_orientador': 'Nenhuma.',
            'relacao_artigo_sbpo': 'não relacionado',
            'relacao_experimento_full': 'documenta',
            'observacoes_defesa': 'Instrumento de conferência rápida.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'DOC01',
        },
    ],
    'Acoes_Revisao': [
        {
            'notebook_id': 'AUD02',
            'ajuste_notebook_necessario': 'não',
            'descricao_ajuste_notebook': 'v1.2 final; contrato 6/13, lift alinhado no teste fixo e R10 não bloqueante documentado.',
            'risco_se_nao_ajustar': 'baixo',
            'descricao_risco': 'Apenas risco de textos externos ainda citarem a faixa legada.',
            'responsavel_acao': 'Sérgio',
            'acao_recomendada': 'Atualizar seletivamente o Mapa da Escrita e documentos ativos.',
            'prioridade_revisao': 'média',
            'status_acao': 'concluída',
            'prazo_sugerido': 'Concluído',
            'notas_adicionais': 'AUD01 v1.6 permanece histórico; AUD02 v1.2 representa a linhagem corrigida.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'AUD02 v1.2',
        },
        {
            'notebook_id': 'DOC01',
            'ajuste_notebook_necessario': 'não',
            'descricao_ajuste_notebook': 'Executar novamente somente quando artefatos ou curadoria durável mudarem.',
            'risco_se_nao_ajustar': 'médio',
            'descricao_risco': 'Matriz v7.0 pode ficar desatualizada em relação aos artefatos.',
            'responsavel_acao': 'Sérgio',
            'acao_recomendada': 'Regenerar a matriz antes de marcos de escrita/defesa.',
            'prioridade_revisao': 'média',
            'status_acao': 'concluída',
            'prazo_sugerido': 'Sob demanda',
            'notas_adicionais': 'A matriz é derivada; não altera o pipeline.',
            'curadoria_v7': 'NOVO_DOC01',
            'fonte_atualizacao_v7': 'DOC01',
        },
    ],
}

for sheet_name, rows in DOC_ROWS.items():
    df = STATIC_SHEETS[sheet_name]
    for row in rows:
        if row['notebook_id'] in set(df['notebook_id'].astype(str)):
            continue
        complete = {col: row.get(col, '') for col in df.columns}
        df = pd.concat([df, pd.DataFrame([complete])], ignore_index=True)
    STATIC_SHEETS[sheet_name] = df

print('Curadoria aplicada.')

# ============================================================
# 4. Registro de artefatos relevantes
# ============================================================

# Correções declarativas de nomes antigos/genéricos presentes na Matriz v5.
REFERENCE_ALIASES = {
    '10_FULL_scenario_decisions_allcells.csv': '10_FULL_scenario_decisions_by_cell.csv',
    '13a_FULL_lstm_tuning_stage1_results_by_cell.csv': '13a_FULL_lstm_tuning_stage1_results_allcells.csv',
    '13a_FULL_lstm_tuning_stage2_results_by_cell.csv': '13a_FULL_lstm_tuning_stage2_results_allcells.csv',
    'final.md': '99_kaggle_reference_final.md',
    'winner_model.json': '11_winner_model.json',
    '16_FULL_artifact_inventory.csv': '16_FULL_artifact_inventory.csv',
    '16_FULL_inventory_summary.json': '16_FULL_nb16_summary.json',
}

REFERENCE_CLASSIFICATION = {
    'ciclo_dsr.pdf': 'PLANNED_NOT_GENERATED',
    'evolucao_metodologica.pdf': 'PLANNED_NOT_GENERATED',
    'framework_completo.pdf': 'PLANNED_NOT_GENERATED',
    'NB99_ABC_Historico.zip': 'ARCHIVED_ELSEWHERE',
    'NB99_ABC_Saneado.zip': 'ARCHIVED_ELSEWHERE',

    # Entradas e artefatos model-facing mantidos deliberadamente fora de
    # 04-reports. O inventário de reports não deve transformá-los em erro.
    '99_C_model_facing_manifest_sha256.csv': 'REFERENCED_OUTSIDE_REPORTS',
    'anticipation_supervised_dataset.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'borg_traces_data.csv': 'REFERENCED_OUTSIDE_REPORTS',
    'episodes_detected.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'google_trace_clean.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'google_trace_clean_keep_hour0.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'trace_raw_validated.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'transition_dataset.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_base.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_features.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_labeled.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_series.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_series_allcells_model_facing_000000000000.parquet': 'REFERENCED_OUTSIDE_REPORTS',
    'window_5min_series_scored.parquet': 'REFERENCED_OUTSIDE_REPORTS',
}

# Figuras distintas que compartilham basename. Devem permanecer como
# artefatos diferentes, identificados pela subpasta.
KNOWN_PATH_DISTINCT_ARTIFACTS = {
    'fig_01_roc_comparativa.png': {
        'NB08_ANTICIPATION': 'figures_nb08_anticipation/fig_01_roc_comparativa.png',
        'NB08_TRANSITION': 'figures_nb08_transition/fig_01_roc_comparativa.png',
    },
    'fig_02_pr_comparativa.png': {
        'NB08_ANTICIPATION': 'figures_nb08_anticipation/fig_02_pr_comparativa.png',
        'NB08_TRANSITION': 'figures_nb08_transition/fig_02_pr_comparativa.png',
    },
}

references: dict[str, dict[str, Any]] = {}


def register_reference(
    filename: str,
    notebook_id: str = '',
    origin: str = '',
    role: str = '',
    path_hint: Any = None,
    description: str = '',
    reference_key: str | None = None,
):
    if not filename:
        return
    original_filename = normalize_filename_token(filename)
    canonical_filename = REFERENCE_ALIASES.get(original_filename, original_filename)
    key = reference_key or canonical_filename
    item = references.setdefault(key, {
        'reference_key': key,
        'filename': canonical_filename,
        'original_filenames': set(),
        'notebook_ids': set(),
        'origins': set(),
        'roles': set(),
        'path_hints': set(),
        'descriptions': set(),
    })
    item['original_filenames'].add(original_filename)
    if original_filename != canonical_filename:
        item['descriptions'].add(
            f'Referência antiga normalizada: {original_filename} → {canonical_filename}.'
        )
        item['origins'].add('DOC01_ALIAS_NORMALIZATION')
    if notebook_id:
        item['notebook_ids'].add(notebook_id)
    if origin:
        item['origins'].add(origin)
    if role:
        item['roles'].add(role)
    if path_hint:
        item['path_hints'].add(str(path_hint))
    if description:
        item['descriptions'].add(description)


# 4.1 Fontes oficialmente inventariadas pelo AUD02.
for _, row in AUD02_SOURCES.iterrows():
    register_reference(
        filename=safe_text(row['filename']),
        notebook_id=safe_text(row['stage']),
        origin='AUD02_SOURCE_INVENTORY',
        role='fonte auditada',
        path_hint=row['path'],
        description=safe_text(row['description']),
    )

# 4.2 Saídas do próprio AUD02.
for path in sorted(AUD02_DIR.iterdir()):
    if path.is_file():
        register_reference(
            filename=path.name,
            notebook_id='AUD02',
            origin='AUD02_OUTPUT',
            role='artefato de auditoria',
            path_hint=path,
            description='Saída da auditoria transversal.',
        )

# 4.3 Referências textuais da Matriz v5.
for sheet_name, df in STATIC_SHEETS.items():
    for _, row in df.iterrows():
        notebook_id = safe_text(row.get('notebook_id', ''))
        for value in row.tolist():
            for filename in extract_filenames(value):
                register_reference(
                    filename=filename,
                    notebook_id=notebook_id or infer_notebook_id(
                        REFERENCE_ALIASES.get(filename, filename)
                    ),
                    origin=f'V5_{sheet_name}',
                    role='referenciado na matriz v5',
                )

# Separar basenames que representam artefatos diferentes por subpasta.
for filename, variants in KNOWN_PATH_DISTINCT_ARTIFACTS.items():
    base = references.pop(filename, None)
    if base is None:
        base = {
            'reference_key': filename,
            'filename': filename,
            'original_filenames': {filename},
            'notebook_ids': {'NB08'},
            'origins': {'DOC01_KNOWN_PATH_DISTINCT'},
            'roles': {'figura NB08'},
            'path_hints': set(),
            'descriptions': set(),
        }
    for qualifier, relative_path in variants.items():
        key = f'{filename}::{qualifier}'
        item = copy.deepcopy(base)
        item['reference_key'] = key
        item['path_hints'] = {relative_path}
        item['origins'].add('DOC01_KNOWN_PATH_DISTINCT')
        item['descriptions'].add(
            f'Artefato distinto preservado pela subpasta: {relative_path}.'
        )
        references[key] = item

# 4.4 Artefatos de fechamento que devem estar na vista transversal.
EXTRA_ARTIFACTS = [
    ('NB16_FULL', '16_FULL_dissertation_figures_registry.csv', 'registro de figuras'),
    ('NB16_FULL', '16_FULL_decision_log.csv', 'log de decisões'),
    ('NB16_FULL', '16_FULL_limitations.csv', 'limitações'),
    ('NB16_FULL', '16_FULL_artifact_manifest_sha256.csv', 'manifesto do ramo'),
    ('NB16_FULL', '16_FULL_nb16_summary.json', 'síntese do inventário'),
    ('NB16_FULL', '16_FULL_artifact_inventory.csv', 'inventário do ramo'),
    ('NB14_FULL', '14_FULL_scenario_summary_by_cell.csv', 'síntese operacional por célula'),
    ('NB13a_FULL', '13a_FULL_lstm_tuning_summary.json', 'síntese do tuning LSTM'),
    ('NB13a_FULL', '13a_FULL_lstm_tuning_stage1_results_allcells.csv', 'resultados agregados do estágio 1'),
    ('NB13a_FULL', '13a_FULL_lstm_tuning_stage2_results_allcells.csv', 'resultados agregados do estágio 2'),
    ('NB12_FULL', '12_FULL_train_p95_review_by_cell.csv', 'revisão TRAIN_P95'),
    ('NB10_FULL', '10_FULL_scenario_decisions_by_cell.csv', 'decisões de avanço por cenário'),
    ('NB06_FULL', '06_FULL_scenario_summary_table.csv', 'rotulagem e prevalência por cenário'),
]
for nid, filename, role in EXTRA_ARTIFACTS:
    register_reference(filename, nid, 'DOC01_EXPLICIT', role)

# Claims por artefato.
claim_by_filename: dict[str, set[str]] = defaultdict(set)
for _, row in AUD02_CLAIMS.iterrows():
    claim_id = safe_text(row['claim_id'])
    for filename in extract_filenames(row.get('evidence_files', '')):
        canonical_filename = REFERENCE_ALIASES.get(filename, filename)
        claim_by_filename[canonical_filename].add(claim_id)

# Fallback físico fora de 04-reports, especialmente pacotes históricos.
search_roots = [
    PROJECT_ROOT / '02-datasets',
    PROJECT_ROOT / '03-models',
    MYDRIVE_ROOT / 'Colab Notebooks',
    PROJECT_ROOT,
    LOCAL_ROOT,
]
FILE_INDEX = make_file_index(search_roots)

# Mapa de pipeline e descrição do notebook.
index_meta = STATIC_SHEETS['Indice_Notebooks'].copy()
index_meta['notebook_id'] = index_meta['notebook_id'].astype(str)
pipeline_map = index_meta.set_index('notebook_id')['pipeline'].to_dict()
notebook_name_map = index_meta.set_index('notebook_id')['notebook_nome'].to_dict()

source_inventory_map = {
    safe_text(row['filename']): row
    for _, row in AUD02_SOURCES.iterrows()
}
source_hash_map = {
    safe_text(row['path']): safe_text(row['sha256'])
    for _, row in AUD02_SOURCE_HASHES.iterrows()
}


def infer_producer_notebook_id(filename: str, relative_path: str = '') -> str:
    producer = infer_notebook_id(filename)
    if producer:
        return producer
    normalized = safe_text(relative_path).replace('\\', '/').upper()
    if '/99_A_' in '/' + normalized or normalized.startswith('99_A_'):
        return 'NB99_A'
    if '/99_B_' in '/' + normalized or normalized.startswith('99_B_'):
        return 'NB99_B'
    if '/99_C_' in '/' + normalized or normalized.startswith('99_C_'):
        return 'NB99_C'
    return ''


def relationship_type(
    producer_notebook_id: str,
    related_notebook_id: str,
    origins: set[str],
) -> str:
    if producer_notebook_id and producer_notebook_id == related_notebook_id:
        return 'PRODUCED_BY'
    if related_notebook_id == 'AUD02' or 'AUD02_SOURCE_INVENTORY' in origins:
        return 'AUDITED_BY'
    if producer_notebook_id and related_notebook_id:
        return 'REFERENCED_BY'
    return 'INDEXED_FOR'


artifact_rows = []
for reference_key, ref in sorted(references.items()):
    filename = ref['filename']
    notebook_ids = sorted(ref['notebook_ids']) or [infer_notebook_id(filename)]
    declared_class = REFERENCE_CLASSIFICATION.get(filename, '')
    exact_path = None
    inventory_record = None
    resolution_status = ''
    resolution_confidence = 'LOW'
    candidate_count = 0
    distinct_hash_count = 0
    candidate_paths = ''

    # Caminhos absolutos registrados pelo AUD02 ou overrides explícitos.
    absolute_hints = []
    relative_hints = []
    for hint in sorted(ref['path_hints']):
        mapped = remap_project_path(hint)
        if mapped is not None and mapped.exists():
            absolute_hints.append(mapped)
        else:
            normalized_hint = safe_text(hint).replace('\\', '/')
            marker = '/04-reports/'
            if marker.lower() in normalized_hint.lower():
                normalized_hint = re.split(
                    re.escape(marker), normalized_hint, maxsplit=1, flags=re.IGNORECASE
                )[1]
            relative_hints.append(normalize_relative_path(normalized_hint))

    if declared_class == 'PLANNED_NOT_GENERATED':
        resolution_status = 'DECLARED_PLANNED_NOT_GENERATED'
        resolution_confidence = 'N/A'
    elif declared_class == 'ARCHIVED_ELSEWHERE':
        override = ARCHIVE_PATH_OVERRIDES.get(filename)
        archive_candidates = []
        if override:
            archive_candidates.append(Path(override).expanduser())
        archive_candidates.extend(FILE_INDEX.get(filename, []))
        archive_candidates = [p for p in deduplicate_paths(archive_candidates) if p.exists()]
        if archive_candidates:
            exact_path = choose_preferred(archive_candidates, exact_name=filename)
            resolution_status = 'ARCHIVE_PATH_FOUND'
            resolution_confidence = 'HIGH'
        else:
            resolution_status = 'DECLARED_ARCHIVED_ELSEWHERE'
            resolution_confidence = 'MEDIUM'
    elif declared_class == 'REFERENCED_OUTSIDE_REPORTS':
        external_candidates = [
            p for p in deduplicate_paths(FILE_INDEX.get(filename, []))
            if p.exists()
        ]
        if external_candidates:
            exact_path = choose_preferred(
                external_candidates, exact_name=filename
            )
            resolution_status = 'OUTSIDE_REPORTS_PATH_FOUND'
            resolution_confidence = 'HIGH'
        else:
            resolution_status = 'DECLARED_OUTSIDE_REPORTS'
            resolution_confidence = 'MEDIUM'
    elif absolute_hints:
        exact_path = absolute_hints[0]
        resolution_status = 'PATH_HINT_MATCH'
        resolution_confidence = 'HIGH'
    else:
        decision = choose_inventory_candidate(
            filename,
            INVENTORY_INDEX.get(filename, []),
            notebook_ids,
            relative_hints,
        )
        inventory_record = decision['record']
        resolution_status = decision['resolution_status']
        resolution_confidence = decision['resolution_confidence']
        candidate_count = decision['candidate_count']
        distinct_hash_count = decision['distinct_hash_count']
        candidate_paths = decision['candidate_paths']
        if inventory_record is not None:
            exact_path = inventory_record['full_path']

    filesystem_present = bool(exact_path and exact_path.exists())
    inventory_present = inventory_record is not None
    exists = filesystem_present or inventory_present

    if filesystem_present:
        size_bytes = exact_path.stat().st_size
        modified_at = datetime.fromtimestamp(
            exact_path.stat().st_mtime, tz=timezone.utc
        ).isoformat()
        sha256, hash_status = sha256_file(exact_path)
        sha256 = sha256 or ''
    elif inventory_record is not None:
        size_bytes = inventory_record.get('size_bytes')
        modified_at = inventory_record.get('last_write_time', '')
        sha256 = safe_text(inventory_record.get('sha256')).lower()
        hash_status = 'FROM_REPORTS_INVENTORY'
    else:
        size_bytes = None
        modified_at = ''
        sha256 = ''
        hash_status = 'NOT_COMPUTED'

    source_row = source_inventory_map.get(filename)
    expected_sha = safe_text(source_row['manifest_sha256']) if source_row is not None else ''
    if not expected_sha and source_row is not None:
        expected_sha = source_hash_map.get(safe_text(source_row['path']), '')

    if expected_sha and sha256:
        hash_match = sha256 == expected_sha
    else:
        hash_match = None

    if declared_class == 'PLANNED_NOT_GENERATED':
        validation_level = 'PLANNED_NOT_GENERATED'
    elif declared_class == 'ARCHIVED_ELSEWHERE':
        validation_level = 'ARCHIVED_ELSEWHERE'
    elif declared_class == 'REFERENCED_OUTSIDE_REPORTS':
        validation_level = 'REFERENCED_OUTSIDE_REPORTS'
    elif filename in source_inventory_map or filename.startswith('AUD02_'):
        validation_level = 'AUD02_AUDITED'
    elif exists:
        validation_level = 'ARTIFACT_INDEXED'
    else:
        validation_level = 'UNRESOLVED_REFERENCE'

    relative_path = (
        inventory_record.get('relative_path', '')
        if inventory_record is not None
        else ''
    )
    producer = infer_producer_notebook_id(filename, relative_path)

    for notebook_id in notebook_ids:
        qualifier = (
            reference_key.split('::', 1)[1]
            if '::' in reference_key
            else ''
        )
        artifact_id = f"{notebook_id or 'UNASSIGNED'}::{filename}"
        if qualifier:
            artifact_id += f'::{qualifier}'
        artifact_rows.append({
            'artifact_id': artifact_id,
            'producer_notebook_id': producer,
            'related_notebook_id': notebook_id,
            'relationship_type': relationship_type(
                producer, notebook_id, ref['origins']
            ),
            'notebook_id': notebook_id,  # compatibilidade com a v1.1
            'notebook_name': safe_text(notebook_name_map.get(notebook_id, '')),
            'pipeline': safe_text(pipeline_map.get(notebook_id, '')),
            'filename': filename,
            'original_reference_names': ' | '.join(sorted(ref['original_filenames'])),
            'role': ' | '.join(sorted(ref['roles'])),
            'description': ' | '.join(sorted(ref['descriptions'])),
            'relative_path': relative_path,
            'path_resolved': str(exact_path) if exact_path else '',
            'exists': exists,
            'inventory_present': inventory_present,
            'filesystem_present': filesystem_present,
            'resolution_status': resolution_status,
            'resolution_confidence': resolution_confidence,
            'candidate_count': candidate_count,
            'distinct_hash_count': distinct_hash_count,
            'candidate_paths': candidate_paths,
            'file_type': (
                exact_path.suffix.lower().lstrip('.')
                if exact_path
                else Path(filename).suffix.lower().lstrip('.')
            ),
            'size_bytes': size_bytes,
            'modified_at_utc': modified_at,
            'sha256_current': sha256,
            'hash_status': hash_status,
            'sha256_expected': expected_sha,
            'hash_match': hash_match,
            'claim_ids': ' | '.join(sorted(claim_by_filename.get(filename, set()))),
            'validation_level': validation_level,
            'reference_origin': ' | '.join(sorted(ref['origins'])),
        })

ARTIFACTS = pd.DataFrame(artifact_rows).drop_duplicates(
    subset=['artifact_id']
).reset_index(drop=True)

print('Artefatos registrados:', len(ARTIFACTS))
print('Níveis:', ARTIFACTS['validation_level'].value_counts(dropna=False).to_dict())
print('Confiança:', ARTIFACTS['resolution_confidence'].value_counts(dropna=False).to_dict())

# ============================================================
# 5. Números atuais, protocolos e visão FULL por célula
# ============================================================

metric_rows: list[dict[str, Any]] = []


def append_metric(
    numero_id: str,
    notebook_id: str,
    metric: str,
    value: Any,
    unit: str,
    protocol: str,
    artifact_filename: str,
    field: str,
    selector: str = '',
    scope: str = 'global',
    cell_id: str = '',
    scenario_label: str = '',
    claim_id: str = '',
    validation_level: str = 'ARTIFACT_INDEXED',
    interpretation: str = '',
    caution: str = '',
):
    artifact_match = ARTIFACTS[
        (ARTIFACTS['filename'] == artifact_filename)
        & (ARTIFACTS['notebook_id'].astype(str) == notebook_id)
    ]
    if artifact_match.empty:
        artifact_match = ARTIFACTS[ARTIFACTS['filename'] == artifact_filename]
    path = artifact_match.iloc[0]['path_resolved'] if not artifact_match.empty else ''
    sha = artifact_match.iloc[0]['sha256_current'] if not artifact_match.empty else ''
    pipeline = safe_text(pipeline_map.get(notebook_id, ''))

    numeric_value = None
    text_value = ''
    if isinstance(value, (bool, np.bool_)):
        text_value = str(bool(value))
    elif isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        numeric_value = float(value)
    else:
        text_value = safe_text(value)

    metric_rows.append({
        'numero_id': numero_id,
        'notebook_id': notebook_id,
        'pipeline': pipeline,
        'scope': scope,
        'cell_id': cell_id,
        'scenario_label': scenario_label,
        'metric': metric,
        'value_numeric': numeric_value,
        'value_text': text_value,
        'unit': unit,
        'protocol': protocol,
        'artifact_filename': artifact_filename,
        'artifact_path': path,
        'field': field,
        'selector': selector,
        'claim_id': claim_id,
        'validation_level': validation_level,
        'source_sha256': sha,
        'interpretation': interpretation,
        'caution': caution,
    })


headline = AUD02_SUMMARY['headline_results']
append_metric('AUD02_observed_cells', 'AUD02', 'células observadas', len(headline['observed_cells']), 'células', 'auditoria', 'AUD02_summary.json', 'headline_results.observed_cells', claim_id='4.1', validation_level='AUD02_AUDITED')
append_metric('AUD02_modelable_cells', 'AUD02', 'células modeláveis', len(headline['modelable_cells']), 'células', 'gate de modelabilidade', 'AUD02_summary.json', 'headline_results.modelable_cells', claim_id='4.1', validation_level='AUD02_AUDITED')
append_metric('FULL_roc_auc_min', 'NB11_FULL', 'ROC-AUC TSCV mínimo soberano', headline['roc_auc_min'], 'proporção', 'TSCV', '11_FULL_metrics_summary_by_cell.csv', 'roc_auc_tscv_mean', 'is_sovereign_winner=True; modelable cells', claim_id='4.2', validation_level='AUD02_AUDITED')
append_metric('FULL_roc_auc_max', 'NB11_FULL', 'ROC-AUC TSCV máximo soberano', headline['roc_auc_max'], 'proporção', 'TSCV', '11_FULL_metrics_summary_by_cell.csv', 'roc_auc_tscv_mean', 'is_sovereign_winner=True; modelable cells', claim_id='4.2', validation_level='AUD02_AUDITED')
append_metric('FULL_lift_min', 'AUD02', 'lift de PR-AUC mínimo oficial', headline['lift_min'], 'vezes', 'teste fixo herdado', 'AUD02_pr_auc_lift_fixed_test_by_cell.csv', 'lift_pr_fixed_test', 'modelable cells', claim_id='4.3', validation_level='AUD02_AUDITED')
append_metric('FULL_lift_max', 'AUD02', 'lift de PR-AUC máximo oficial', headline['lift_max'], 'vezes', 'teste fixo herdado', 'AUD02_pr_auc_lift_fixed_test_by_cell.csv', 'lift_pr_fixed_test', 'modelable cells', claim_id='4.3', validation_level='AUD02_AUDITED')
append_metric('FULL_anticipation_median', 'NB14_FULL', 'mediana da antecipação episódica', headline['anticipation_median'], 'proporção', 'teste fixo herdado; τ=0,5', 'AUD02_anticipation_claim_audit.csv', 'median_episode_anticipation_rate', claim_id='4.4', validation_level='AUD02_AUDITED')
append_metric('FULL_above_reference_count', 'NB11_FULL', 'células modeláveis acima da referência ROC-AUC', headline['n_above_reference'], 'células', 'TSCV; comparação descritiva', 'AUD02_kaggle_comparison.csv', 'supera_referencia', 'is_modelable=True', claim_id='4.9', validation_level='AUD02_AUDITED')
append_metric('FULL_lstm_nominal_cells', 'NB13a_FULL', 'células que passam somente pelo gate nominal LSTM', ', '.join(headline['lstm_governance']['nominal_delta_gate_cells']), 'lista', 'TSCV', 'AUD02_lstm_governance_summary.json', 'nominal_delta_gate_cells', claim_id='4.7', validation_level='AUD02_AUDITED')
append_metric('FULL_lstm_promoted_cells', 'NB13a_FULL', 'células com LSTM promovida', 'nenhuma', 'lista', 'TSCV + governança NB14', 'AUD02_lstm_governance_summary.json', 'promoted_lstm_primary_cells', claim_id='4.7', validation_level='AUD02_AUDITED')

# Modelabilidade: todas as oito células.
for _, row in MODELABILITY.iterrows():
    cell = safe_text(row['cell_id'])
    append_metric(
        f'FULL_{cell}_is_modelable', 'NB11_FULL', 'is_modelable', bool(row['is_modelable']), 'booleano',
        'gate de modelabilidade', 'AUD02_modelability_summary.csv', 'is_modelable', f'cell_id={cell}',
        scope='célula', cell_id=cell, scenario_label=safe_text(row['scenario_label']), claim_id='4.1', validation_level='AUD02_AUDITED',
        interpretation=safe_text(row['cell_modelability_status']),
    )
    append_metric(
        f'FULL_{cell}_n_evaluable_episodes', 'NB14_FULL', 'episódios avaliáveis em τ de referência', row['n_evaluable_episodes_tau_reference'], 'episódios',
        'teste fixo herdado; τ=0,5', 'AUD02_modelability_summary.csv', 'n_evaluable_episodes_tau_reference', f'cell_id={cell}',
        scope='célula', cell_id=cell, scenario_label=safe_text(row['scenario_label']), validation_level='AUD02_AUDITED',
    )

# Métricas soberanas TSCV.
for _, row in WINNERS.iterrows():
    cell = safe_text(row['cell_id'])
    common = dict(
        notebook_id='NB11_FULL', protocol='TSCV', artifact_filename='AUD02_nominal_winners_by_cell.csv',
        selector=f'cell_id={cell}; is_sovereign_winner=True', scope='célula', cell_id=cell,
        scenario_label=safe_text(row['scenario_label']), validation_level='AUD02_AUDITED'
    )
    append_metric(f'FULL_{cell}_winner_model', metric='modelo soberano', value=row['model'], unit='categórico', field='model', claim_id='4.6', **common)
    append_metric(f'FULL_{cell}_f1_tscv', metric='F1 médio soberano', value=row['f1_tscv_mean'], unit='proporção', field='f1_tscv_mean', **common)
    append_metric(f'FULL_{cell}_roc_auc_tscv', metric='ROC-AUC médio soberano', value=row['roc_auc_tscv_mean'], unit='proporção', field='roc_auc_tscv_mean', claim_id='4.2/4.9', **common)
    append_metric(f'FULL_{cell}_pr_auc_tscv', metric='PR-AUC média soberana', value=row['average_precision_tscv_mean'], unit='proporção', field='average_precision_tscv_mean', **common)

# Lift oficial alinhado.
for _, row in LIFT_FIXED.iterrows():
    cell = safe_text(row['cell_id'])
    common = dict(
        protocol='teste fixo herdado', artifact_filename='AUD02_pr_auc_lift_fixed_test_by_cell.csv',
        selector=f'cell_id={cell}', scope='célula', cell_id=cell, scenario_label=safe_text(row['scenario_label']),
        claim_id='4.3', validation_level='AUD02_AUDITED'
    )
    append_metric(f'FULL_{cell}_ap_fixed', 'AUD02', 'Average Precision no teste fixo', row['average_precision_score_calibrated'], 'proporção', field='average_precision_score_calibrated', **common)
    append_metric(f'FULL_{cell}_prevalence_fixed', 'AUD02', 'prevalência no teste fixo', row['prevalence_fixed_test'], 'proporção', field='prevalence_fixed_test', **common)
    append_metric(f'FULL_{cell}_lift_fixed', 'AUD02', 'lift de PR-AUC oficial', row['lift_pr_fixed_test'], 'vezes', field='lift_pr_fixed_test', **common)

# Métricas operacionais.
for _, row in OPERATIONAL.iterrows():
    cell = safe_text(row['cell_id'])
    common = dict(
        notebook_id='NB14_FULL', protocol='teste fixo herdado; τ=0,5', artifact_filename='AUD02_operational_metrics_by_cell.csv',
        selector=f'cell_id={cell}', scope='célula', cell_id=cell, scenario_label=safe_text(row['scenario_label']),
        claim_id='4.4', validation_level='AUD02_AUDITED'
    )
    append_metric(f'FULL_{cell}_f1_tau_ref', metric='F1 em τ de referência', value=row['f1_tau_reference'], unit='proporção', field='f1_tau_reference', **common)
    append_metric(f'FULL_{cell}_false_alerts_day', metric='falsos alertas por dia em τ de referência', value=row['false_alerts_per_day_tau_reference'], unit='alertas/dia', field='false_alerts_per_day_tau_reference', **common)
    append_metric(f'FULL_{cell}_anticipation_rate', metric='taxa de antecipação episódica', value=row['episode_anticipation_rate_tau_reference'], unit='proporção', field='episode_anticipation_rate_tau_reference', **common)
    append_metric(f'FULL_{cell}_lead_time_median', metric='lead time mediano', value=row['lead_time_minutes_median_tau_reference'], unit='minutos', field='lead_time_minutes_median_tau_reference', **common)

# Governança LSTM e comparação de referência.
for _, row in MODEL_GOV.iterrows():
    cell = safe_text(row['cell_id'])
    common = dict(
        notebook_id='NB13a_FULL', protocol='TSCV / governança não pareada', artifact_filename='AUD02_model_governance_by_cell.csv',
        selector=f'cell_id={cell}', scope='célula', cell_id=cell, scenario_label=safe_text(row['baseline_scenario']),
        claim_id='4.7', validation_level='AUD02_AUDITED'
    )
    append_metric(f'FULL_{cell}_lstm_delta_f1', metric='ΔF1 LSTM ajustada - NB11', value=row['delta_f1'], unit='proporção', field='delta_f1', **common)
    append_metric(f'FULL_{cell}_lstm_nominal_gate', metric='passa gate nominal ΔF1≥0,03', value=bool(row['nominal_delta_gate_pass']), unit='booleano', field='nominal_delta_gate_pass', **common)
    append_metric(f'FULL_{cell}_lstm_robust_gate', metric='passa gate descritivo ΔF1/SE≥2', value=bool(row['robustness_gate_pass']), unit='booleano', field='robustness_gate_pass', **common)
    append_metric(f'FULL_{cell}_lstm_final_position', metric='posição final da LSTM', value=row['final_lstm_position'], unit='categórico', field='final_lstm_position', **common)

for _, row in KAGGLE_COMPARISON.iterrows():
    cell = safe_text(row['cell_id'])
    append_metric(
        f'FULL_{cell}_delta_roc_reference', 'NB11_FULL', 'Δ ROC-AUC vs. valor de referência', row['delta_roc_auc'], 'proporção',
        'TSCV; comparação descritiva', 'AUD02_kaggle_comparison.csv', 'delta_roc_auc', f'cell_id={cell}',
        scope='célula', cell_id=cell, scenario_label=safe_text(row['scenario_label']), claim_id='4.9', validation_level='AUD02_AUDITED',
        caution='Bases, protocolos e cenários não idênticos; sem teste inferencial.',
    )

METRICS = pd.DataFrame(metric_rows)

# ----------------------------
# FULL por célula
# ----------------------------
full = pd.DataFrame({'cell_id': list('abcdefgh')})

m_cols = [
    'cell_id', 'is_modelable', 'cell_modelability_status', 'n_scenarios',
    'n_modelable_scenarios', 'min_pos_test', 'n_total_episodes',
    'n_evaluable_episodes_tau_reference', 'scenario_label'
]
full = full.merge(MODELABILITY[m_cols], on='cell_id', how='left')

w_cols = [
    'cell_id', 'scenario_label', 'feature_set', 'model', 'n_folds_valid',
    'f1_tscv_mean', 'f1_tscv_std', 'roc_auc_tscv_mean', 'roc_auc_tscv_std',
    'average_precision_tscv_mean', 'average_precision_tscv_std',
    'n_positivos_train', 'n_positivos_test'
]
winners_renamed = WINNERS[w_cols].rename(columns={'scenario_label': 'sovereign_scenario_label'})
full = full.merge(winners_renamed, on='cell_id', how='left')

l_cols = [
    'cell_id', 'prevalence_fixed_test', 'average_precision_score_calibrated',
    'lift_pr_fixed_test', 'protocol_alignment'
]
full = full.merge(LIFT_FIXED[l_cols], on='cell_id', how='left')

o_cols = [
    'cell_id', 'tau_reference', 'f1_tau_reference',
    'false_alerts_per_day_tau_reference', 'episode_anticipation_rate_tau_reference',
    'lead_time_minutes_median_tau_reference', 'n_anticipated_episodes_tau_reference'
]
full = full.merge(OPERATIONAL[o_cols], on='cell_id', how='left')

g_cols = [
    'cell_id', 'lstm_f1', 'delta_f1', 'delta_f1_se_ratio', 'nominal_delta_gate_pass',
    'robustness_gate_pass', 'promoted_as_primary_in_nb14',
    'evaluated_as_secondary_in_nb14', 'final_lstm_position', 'score_source_coherence'
]
full = full.merge(MODEL_GOV[g_cols], on='cell_id', how='left')
full = full.rename(columns={'delta_f1_se_ratio': 'delta_over_se'})

k_cols = ['cell_id', 'benchmark_roc_auc', 'delta_roc_auc', 'supera_referencia']
full = full.merge(KAGGLE_COMPARISON[k_cols], on='cell_id', how='left')

full['scenario_label'] = full['sovereign_scenario_label'].fillna(full['scenario_label'])
full['official_range_inclusion'] = np.where(full['is_modelable'].fillna(False), 'INCLUIR', 'EXCLUIR — não modelável')
full['primary_score_source'] = 'NB11_FULL'
full['lstm_role'] = np.where(
    full['evaluated_as_secondary_in_nb14'].fillna(False),
    'sensibilidade secundária; não promovida',
    np.where(full['is_modelable'].fillna(False), 'não promovida', 'não aplicável')
)
full['validation_level'] = 'AUD02_AUDITED'
full['source_bundle'] = (
    'AUD02_modelability_summary.csv | AUD02_nominal_winners_by_cell.csv | '
    'AUD02_pr_auc_lift_fixed_test_by_cell.csv | AUD02_operational_metrics_by_cell.csv | '
    'AUD02_model_governance_by_cell.csv | AUD02_kaggle_comparison.csv'
)

FULL_BY_CELL = full[[
    'cell_id', 'is_modelable', 'official_range_inclusion', 'cell_modelability_status',
    'n_modelable_scenarios', 'n_evaluable_episodes_tau_reference',
    'scenario_label', 'feature_set', 'model', 'primary_score_source', 'n_positivos_test',
    'f1_tscv_mean', 'roc_auc_tscv_mean', 'average_precision_tscv_mean',
    'average_precision_score_calibrated', 'prevalence_fixed_test', 'lift_pr_fixed_test',
    'tau_reference', 'f1_tau_reference', 'false_alerts_per_day_tau_reference',
    'n_anticipated_episodes_tau_reference', 'episode_anticipation_rate_tau_reference',
    'lead_time_minutes_median_tau_reference', 'delta_f1', 'delta_over_se',
    'nominal_delta_gate_pass', 'robustness_gate_pass', 'promoted_as_primary_in_nb14',
    'lstm_role', 'delta_roc_auc', 'supera_referencia', 'validation_level', 'source_bundle'
]].copy()

# ----------------------------
# Protocolos
# ----------------------------
PROTOCOLS = pd.DataFrame([
    {
        'topic': 'Seleção de modelos',
        'metrics': 'F1, recall e PR-AUC para desempate',
        'protocol': 'TimeSeriesSplit',
        'source': 'NB11_FULL',
        'claim_id': '4.2 / 4.6 / 4.9',
        'interpretation': 'Define o resultado soberano por célula.',
        'caution': 'Não escolher retrospectivamente pelo ROC-AUC.',
    },
    {
        'topic': 'Discriminação soberana',
        'metrics': 'ROC-AUC e PR-AUC TSCV',
        'protocol': 'TimeSeriesSplit',
        'source': 'NB11_FULL',
        'claim_id': '4.2 / 4.9',
        'interpretation': 'Qualidade de ordenação do resultado selecionado por F1.',
        'caution': 'Comparação com o protótipo refinado é descritiva.',
    },
    {
        'topic': 'Lift oficial',
        'metrics': 'AP / prevalência',
        'protocol': 'teste fixo herdado — mesmas observações',
        'source': 'NB11_FULL + AUD02',
        'claim_id': '4.3',
        'interpretation': 'Vezes acima da linha de base da prevalência no mesmo teste.',
        'caution': 'Não usar a faixa legada de protocolo misto.',
    },
    {
        'topic': 'Métricas operacionais',
        'metrics': 'τ, F1, falsos alertas, antecipação e lead time',
        'protocol': 'teste fixo herdado',
        'source': 'NB14_FULL',
        'claim_id': '4.4',
        'interpretation': 'Avaliação operacional por célula.',
        'caution': 'Episódios não avaliáveis não entram no denominador.',
    },
    {
        'topic': 'Governança LSTM',
        'metrics': 'ΔF1 e ΔF1/SE',
        'protocol': 'TSCV; comparação não pareada',
        'source': 'NB13_FULL / NB13a_FULL / NB14_FULL',
        'claim_id': '4.7',
        'interpretation': 'Gate nominal e gate descritivo de robustez.',
        'caution': 'Não é teste de significância estatística.',
    },
    {
        'topic': 'Calibração',
        'metrics': 'Brier e diagnósticos correlatos',
        'protocol': 'diagnóstico complementar',
        'source': 'NB11_FULL',
        'claim_id': '4.8',
        'interpretation': 'Contexto e limitação.',
        'caution': 'Não reabre o escopo experimental.',
    },
])

CLAIMS_VIEW = AUD02_CLAIMS[[
    'claim_id', 'claim_text', 'module', 'status', 'result', 'evidence_files',
    'evidence_fields', 'derivation_rule', 'source_hashes_sha256', 'notes'
]].copy()

# ----------------------------
# Cross-check independente da consolidação contra o AUD02
# ----------------------------
crosscheck_rows: list[dict[str, Any]] = []


def add_crosscheck(
    check_key: str,
    scope: str,
    cell_id: str,
    field: str,
    observed: Any,
    expected: Any,
    tolerance: str = '',
):
    if isinstance(observed, (int, float, np.integer, np.floating)) and isinstance(
        expected, (int, float, np.integer, np.floating)
    ):
        passed = bool(np.isclose(float(observed), float(expected), rtol=1e-10, atol=1e-12))
    else:
        passed = safe_text(observed) == safe_text(expected)
    crosscheck_rows.append({
        'check_key': check_key,
        'scope': scope,
        'cell_id': cell_id,
        'field': field,
        'observed_from_matrix_build': observed,
        'expected_from_aud02': expected,
        'tolerance': tolerance,
        'status': 'PASS' if passed else 'FAIL',
    })


modelable_full = FULL_BY_CELL[FULL_BY_CELL['is_modelable'].fillna(False)].copy()
add_crosscheck(
    'HEADLINE_ROC_MIN', 'headline', '', 'roc_auc_min',
    modelable_full['roc_auc_tscv_mean'].min(), headline['roc_auc_min'],
    'np.isclose(rtol=1e-10, atol=1e-12)',
)
add_crosscheck(
    'HEADLINE_ROC_MAX', 'headline', '', 'roc_auc_max',
    modelable_full['roc_auc_tscv_mean'].max(), headline['roc_auc_max'],
    'np.isclose(rtol=1e-10, atol=1e-12)',
)
add_crosscheck(
    'HEADLINE_LIFT_MIN', 'headline', '', 'lift_min',
    modelable_full['lift_pr_fixed_test'].min(), headline['lift_min'],
    'np.isclose(rtol=1e-10, atol=1e-12)',
)
add_crosscheck(
    'HEADLINE_LIFT_MAX', 'headline', '', 'lift_max',
    modelable_full['lift_pr_fixed_test'].max(), headline['lift_max'],
    'np.isclose(rtol=1e-10, atol=1e-12)',
)
add_crosscheck(
    'HEADLINE_ANTICIPATION_MEDIAN', 'headline', '', 'anticipation_median',
    modelable_full['episode_anticipation_rate_tau_reference'].median(),
    headline['anticipation_median'],
    'np.isclose(rtol=1e-10, atol=1e-12)',
)
add_crosscheck(
    'HEADLINE_MODELABLE_CELLS', 'headline', '', 'modelable_cells',
    ','.join(modelable_full['cell_id'].astype(str).tolist()),
    ','.join(headline['modelable_cells']),
)
add_crosscheck(
    'HEADLINE_ABOVE_REFERENCE', 'headline', '', 'n_above_reference',
    int(modelable_full['supera_referencia'].fillna(False).sum()),
    int(headline['n_above_reference']),
)
add_crosscheck(
    'HEADLINE_LSTM_NOMINAL', 'headline', '', 'nominal_delta_gate_cells',
    ','.join(sorted(MODEL_GOV.loc[
        MODEL_GOV['nominal_delta_gate_pass'].fillna(False), 'cell_id'
    ].astype(str).tolist())),
    ','.join(sorted(headline['lstm_governance']['nominal_delta_gate_cells'])),
)
add_crosscheck(
    'HEADLINE_LSTM_PROMOTED', 'headline', '', 'promoted_cells',
    ','.join(sorted(MODEL_GOV.loc[
        MODEL_GOV['promoted_as_primary_in_nb14'].fillna(False), 'cell_id'
    ].astype(str).tolist())),
    ','.join(sorted(headline['lstm_governance']['promoted_lstm_primary_cells'])),
)

# Conferências por célula: a consolidação não pode trocar modelo, cenário,
# modelabilidade, ROC-AUC, lift ou antecipação.
for _, actual in FULL_BY_CELL.iterrows():
    cell = safe_text(actual['cell_id'])

    expected_modelability = MODELABILITY[
        MODELABILITY['cell_id'].astype(str) == cell
    ].iloc[0]
    add_crosscheck(
        f'CELL_{cell}_MODELABLE', 'cell', cell, 'is_modelable',
        bool(actual['is_modelable']), bool(expected_modelability['is_modelable'])
    )

    winner_rows = WINNERS[WINNERS['cell_id'].astype(str) == cell]
    if not winner_rows.empty:
        expected_winner = winner_rows.iloc[0]
        add_crosscheck(
            f'CELL_{cell}_SCENARIO', 'cell', cell, 'scenario_label',
            actual['scenario_label'], expected_winner['scenario_label']
        )
        add_crosscheck(
            f'CELL_{cell}_MODEL', 'cell', cell, 'model',
            actual['model'], expected_winner['model']
        )
        add_crosscheck(
            f'CELL_{cell}_ROC', 'cell', cell, 'roc_auc_tscv_mean',
            actual['roc_auc_tscv_mean'], expected_winner['roc_auc_tscv_mean'],
            'np.isclose(rtol=1e-10, atol=1e-12)',
        )

    lift_rows = LIFT_FIXED[LIFT_FIXED['cell_id'].astype(str) == cell]
    if not lift_rows.empty:
        expected_lift = lift_rows.iloc[0]
        add_crosscheck(
            f'CELL_{cell}_LIFT', 'cell', cell, 'lift_pr_fixed_test',
            actual['lift_pr_fixed_test'], expected_lift['lift_pr_fixed_test'],
            'np.isclose(rtol=1e-10, atol=1e-12)',
        )

    operational_rows = OPERATIONAL[OPERATIONAL['cell_id'].astype(str) == cell]
    if not operational_rows.empty:
        expected_operational = operational_rows.iloc[0]
        add_crosscheck(
            f'CELL_{cell}_ANTICIPATION', 'cell', cell,
            'episode_anticipation_rate_tau_reference',
            actual['episode_anticipation_rate_tau_reference'],
            expected_operational['episode_anticipation_rate_tau_reference'],
            'np.isclose(rtol=1e-10, atol=1e-12)',
        )

NUMERIC_CROSSCHECK = pd.DataFrame(crosscheck_rows)
print('Métricas atuais:', len(METRICS))
print('FULL por célula:', FULL_BY_CELL.shape)
print('Protocolos:', len(PROTOCOLS))

# ============================================================
# 6. Validação prévia da geração
# ============================================================

validation_rows: list[dict[str, Any]] = []


def add_validation(check_id: str, description: str, observed: Any, expected: Any, status: str, action: str = ''):
    validation_rows.append({
        'check_id': check_id,
        'description': description,
        'observed': safe_text(observed),
        'expected': safe_text(expected),
        'status': status,
        'action': action,
    })


add_validation('D01', 'Matriz v5 localizada.', SOURCE_MATRIX_PATH, 'arquivo existente', 'PASS' if SOURCE_MATRIX_PATH.exists() else 'FAIL')
add_validation('D02', 'AUD02 v1.2 sem falha material; REVIEW de regressão é permitido quando não bloqueante.', AUD02_SUMMARY.get('overall_status'), 'status não iniciado por FAIL', 'PASS' if not str(AUD02_SUMMARY.get('overall_status', '')).startswith('FAIL') else 'FAIL')
add_validation('D03', 'Todos os checks do AUD02 aprovados sem falha material.', AUD02_CHECKS['status'].value_counts().to_dict(), 'nenhum FAIL', 'PASS' if not AUD02_CHECKS['status'].astype(str).str.startswith('FAIL').any() else 'FAIL')
add_validation('D04', 'FULL por célula contém a–h.', FULL_BY_CELL['cell_id'].tolist(), list('abcdefgh'), 'PASS' if FULL_BY_CELL['cell_id'].tolist() == list('abcdefgh') else 'FAIL')
add_validation('D05', 'Única célula não modelável é d.', FULL_BY_CELL.loc[~FULL_BY_CELL['is_modelable'].fillna(False), 'cell_id'].tolist(), ['d'], 'PASS' if FULL_BY_CELL.loc[~FULL_BY_CELL['is_modelable'].fillna(False), 'cell_id'].tolist() == ['d'] else 'FAIL')
add_validation('D06', 'Lift oficial usa protocolo alinhado.', sorted(LIFT_FIXED['protocol_alignment'].unique().tolist()), ['ALIGNED_NB11_FIXED_INHERITED_TEST'], 'PASS' if set(LIFT_FIXED['protocol_alignment']) == {'ALIGNED_NB11_FIXED_INHERITED_TEST'} else 'FAIL')
add_validation('D07', 'Nenhuma LSTM promovida.', MODEL_GOV.loc[MODEL_GOV['promoted_as_primary_in_nb14'].fillna(False), 'cell_id'].tolist(), [], 'PASS' if not MODEL_GOV['promoted_as_primary_in_nb14'].fillna(False).any() else 'FAIL')
add_validation('D08', 'Artefatos relevantes indexados.', len(ARTIFACTS), '> 0', 'PASS' if len(ARTIFACTS) > 0 else 'FAIL')

unresolved_count = int((ARTIFACTS['validation_level'] == 'UNRESOLVED_REFERENCE').sum())
add_validation(
    'D09',
    'Referências realmente não resolvidas após aliases e classificações.',
    unresolved_count,
    0,
    'PASS' if unresolved_count == 0 else 'REVIEW',
    'Apenas UNRESOLVED_REFERENCE exige inspeção; planejados e arquivos históricos externos são classificados separadamente.'
)

add_validation('D10', 'Hashes divergentes entre fontes auditadas.', int((ARTIFACTS['hash_match'] == False).sum()), 0, 'PASS' if not (ARTIFACTS['hash_match'] == False).any() else 'FAIL')
add_validation('D11', 'Números atuais têm fonte e campo.', int(((METRICS['artifact_filename'] == '') | (METRICS['field'] == '')).sum()), 0, 'PASS' if not ((METRICS['artifact_filename'] == '') | (METRICS['field'] == '')).any() else 'FAIL')
add_validation('D12', 'Prosa durável preservada.', {k: len(v) for k, v in STATIC_SHEETS.items()}, 'abas estáticas não vazias', 'PASS' if all(len(v) > 0 for v in STATIC_SHEETS.values()) else 'FAIL')

STALE_PATTERNS = [
    r'1,6×[^\n]{0,20}7,5×',
    r'6 das 8 células',
    r'6 de 8 células',
    r'7 das 8 células',
    r'1\.159 artefatos',
    r'18 OK, 1 WARN',
]
stale_hits = []
for sheet_name, df in STATIC_SHEETS.items():
    for column in df.columns:
        for idx, value in df[column].items():
            if not isinstance(value, str):
                continue
            for stale_pattern in STALE_PATTERNS:
                if re.search(stale_pattern, value, flags=re.IGNORECASE):
                    stale_hits.append(f'{sheet_name}:{idx + 2}:{column}:{stale_pattern}')
add_validation('D13', 'Formulações críticas antigas removidas da curadoria ativa.', len(stale_hits), 0, 'PASS' if not stale_hits else 'REVIEW', ' | '.join(stale_hits[:20]))

crosscheck_failures = NUMERIC_CROSSCHECK[
    NUMERIC_CROSSCHECK['status'] != 'PASS'
]
add_validation(
    'D14',
    'Cross-check numérico e categórico da consolidação contra o AUD02.',
    int(len(crosscheck_failures)),
    0,
    'PASS' if crosscheck_failures.empty else 'FAIL',
    ' | '.join(crosscheck_failures['check_key'].astype(str).tolist()[:30]),
)

low_confidence = ARTIFACTS[
    ARTIFACTS['resolution_confidence'] == 'LOW'
]
add_validation(
    'D15',
    'Resoluções de artefato com confiança LOW.',
    int(len(low_confidence)),
    0,
    'PASS' if low_confidence.empty else 'REVIEW',
    ' | '.join(low_confidence['artifact_id'].astype(str).tolist()[:30]),
)

canonical_hits = []
for sheet_name, df in STATIC_SHEETS.items():
    for column in df.columns:
        for idx, value in df[column].items():
            if isinstance(value, str) and re.search(r'\bcanônic[oa]s?\b', value, flags=re.IGNORECASE):
                canonical_hits.append(f'{sheet_name}:{idx + 2}:{column}')
add_validation(
    'D16',
    'Termo “canônico” ausente das abas curadas ativas.',
    len(canonical_hits),
    0,
    'PASS' if not canonical_hits else 'REVIEW',
    ' | '.join(canonical_hits[:30]),
)

known_distinct = ARTIFACTS[
    ARTIFACTS['reference_origin'].fillna('').str.contains(
        'DOC01_KNOWN_PATH_DISTINCT', regex=False
    )
]
add_validation(
    'D17',
    'Figuras NB08 de mesmo basename preservadas como artefatos distintos.',
    int(len(known_distinct)),
    4,
    'PASS' if len(known_distinct) == 4 else 'FAIL',
)

classified_absences = ARTIFACTS[
    ARTIFACTS['validation_level'].isin([
        'PLANNED_NOT_GENERATED', 'ARCHIVED_ELSEWHERE'
    ])
]
add_validation(
    'D18',
    'Ausências conhecidas classificadas explicitamente.',
    classified_absences['validation_level'].value_counts().to_dict(),
    {'PLANNED_NOT_GENERATED': 3, 'ARCHIVED_ELSEWHERE': 2},
    'PASS' if (
        int((classified_absences['validation_level'] == 'PLANNED_NOT_GENERATED').sum()) == 3
        and int((classified_absences['validation_level'] == 'ARCHIVED_ELSEWHERE').sum()) == 2
    ) else 'REVIEW',
)

VALIDATION = pd.DataFrame(validation_rows)

if (VALIDATION['status'] == 'FAIL').any():
    OVERALL_STATUS = 'FAIL'
elif (VALIDATION['status'] == 'REVIEW').any():
    OVERALL_STATUS = 'PASS_WITH_REVIEW'
else:
    OVERALL_STATUS = 'PASS'

if OVERALL_STATUS == 'FAIL':
    display(VALIDATION)
    display(NUMERIC_CROSSCHECK)
    raise RuntimeError('DOC01 interrompido: há validações FAIL antes da geração da matriz.')

display(VALIDATION)

# ============================================================
# 7. Geração da Matriz v7
# ============================================================

HEADER_FILL = PatternFill('solid', fgColor='1F4E78')
HEADER_FONT = Font(color='FFFFFF', bold=True)
TITLE_FILL = PatternFill('solid', fgColor='17365D')
TITLE_FONT = Font(color='FFFFFF', bold=True, size=16)
SUBTITLE_FILL = PatternFill('solid', fgColor='D9EAF7')
THIN_GRAY = Side(style='thin', color='D9E1F2')
CELL_BORDER = Border(left=THIN_GRAY, right=THIN_GRAY, top=THIN_GRAY, bottom=THIN_GRAY)
PASS_FILL = PatternFill('solid', fgColor='C6EFCE')
REVIEW_FILL = PatternFill('solid', fgColor='FFEB9C')
FAIL_FILL = PatternFill('solid', fgColor='FFC7CE')


def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^A-Za-z0-9_]', '_', name)
    if not clean or clean[0].isdigit():
        clean = 'T_' + clean
    return clean[:200]


def remove_sheet_if_exists(wb, name: str):
    if name in wb.sheetnames:
        wb.remove(wb[name])


def write_dataframe_sheet(wb, name: str, df: pd.DataFrame, table_name: str | None = None, freeze: str = 'A2'):
    remove_sheet_if_exists(wb, name)
    ws = wb.create_sheet(name)
    ws.freeze_panes = freeze
    clean = df.copy()
    clean = clean.replace({np.nan: None})
    headers = list(clean.columns)
    ws.append(headers)
    for row in clean.itertuples(index=False, name=None):
        ws.append(list(row))

    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = CELL_BORDER
    ws.row_dimensions[1].height = 34

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='top', wrap_text=True)
            cell.border = CELL_BORDER

    if ws.max_row >= 2 and ws.max_column >= 1:
        ref = f'A1:{get_column_letter(ws.max_column)}{ws.max_row}'
        table = Table(displayName=sanitize_table_name(table_name or f'T_{name}'), ref=ref)
        table.tableStyleInfo = TableStyleInfo(
            name='TableStyleMedium2', showFirstColumn=False, showLastColumn=False,
            showRowStripes=True, showColumnStripes=False,
        )
        ws.add_table(table)
        ws.auto_filter.ref = ref

    # Larguras com teto para texto longo.
    for idx, column in enumerate(headers, start=1):
        values = [safe_text(column)] + [safe_text(ws.cell(r, idx).value) for r in range(2, min(ws.max_row, 120) + 1)]
        max_len = max((len(v) for v in values), default=10)
        if any(token in column.lower() for token in ['description', 'observ', 'interpret', 'caution', 'path', 'rule', 'notes', 'question', 'response', 'amea', 'mitig', 'achad', 'logica', 'objetivo', 'artefatos']):
            width = min(max(max_len * 0.75, 20), 55)
        else:
            width = min(max(max_len * 0.95, 10), 28)
        ws.column_dimensions[get_column_letter(idx)].width = width

    # Formatos numéricos por semântica.
    for idx, column in enumerate(headers, start=1):
        lc = column.lower()
        if any(token in lc for token in ['f1_', 'roc_auc', 'precision', 'recall', 'prevalence', 'anticipation_rate', 'delta_f1', 'delta_roc', 'brier']):
            for cell in ws.iter_cols(min_col=idx, max_col=idx, min_row=2, max_row=ws.max_row):
                for c in cell:
                    if isinstance(c.value, (int, float)):
                        c.number_format = '0.000000'
        elif 'lift' in lc or 'false_alerts' in lc or 'lead_time' in lc:
            for cell in ws.iter_cols(min_col=idx, max_col=idx, min_row=2, max_row=ws.max_row):
                for c in cell:
                    if isinstance(c.value, (int, float)):
                        c.number_format = '0.000'
        elif lc in {'size_bytes'}:
            for cell in ws.iter_cols(min_col=idx, max_col=idx, min_row=2, max_row=ws.max_row):
                for c in cell:
                    if isinstance(c.value, (int, float)):
                        c.number_format = '#,##0'

    # Hyperlink local/Drive quando o caminho existe.
    for header in ['path_resolved', 'artifact_path']:
        if header in headers:
            col_idx = headers.index(header) + 1
            for r in range(2, ws.max_row + 1):
                cell = ws.cell(r, col_idx)
                if cell.value:
                    cell.hyperlink = str(cell.value)
                    cell.style = 'Hyperlink'
                    cell.alignment = Alignment(vertical='top', wrap_text=True)
    return ws


def add_status_formatting(ws, header_name: str):
    headers = [c.value for c in ws[1]]
    if header_name not in headers:
        return
    col = headers.index(header_name) + 1
    col_letter = get_column_letter(col)
    last = ws.max_row
    if last < 2:
        return
    ws.conditional_formatting.add(
        f'{col_letter}2:{col_letter}{last}',
        FormulaRule(formula=[f'ISNUMBER(SEARCH("PASS",{col_letter}2))'], fill=PASS_FILL)
    )
    ws.conditional_formatting.add(
        f'{col_letter}2:{col_letter}{last}',
        FormulaRule(formula=[f'ISNUMBER(SEARCH("REVIEW",{col_letter}2))'], fill=REVIEW_FILL)
    )
    ws.conditional_formatting.add(
        f'{col_letter}2:{col_letter}{last}',
        FormulaRule(formula=[f'ISNUMBER(SEARCH("FAIL",{col_letter}2))'], fill=FAIL_FILL)
    )


# Carregar a v5 para preservar seus estilos nas abas curadas.
wb = load_workbook(SOURCE_MATRIX_PATH)

# Remover todas as tabelas das abas estáticas antes de reescrever, evitando refs antigas.
for sheet_name, df in STATIC_SHEETS.items():
    if sheet_name not in wb.sheetnames:
        ws = wb.create_sheet(sheet_name)
    else:
        ws = wb[sheet_name]
    for table_name in list(ws.tables.keys()):
        del ws.tables[table_name]
    ws.delete_rows(1, ws.max_row)
    for row in [list(df.columns)] + df.replace({np.nan: None}).values.tolist():
        ws.append(row)
    ws.freeze_panes = 'A2'
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = CELL_BORDER
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='top', wrap_text=True)
            cell.border = CELL_BORDER
    ref = f'A1:{get_column_letter(ws.max_column)}{ws.max_row}'
    if ws.max_row >= 2:
        table = Table(displayName=sanitize_table_name(f'T_{sheet_name}_v7_1'), ref=ref)
        table.tableStyleInfo = TableStyleInfo(name='TableStyleMedium2', showRowStripes=True)
        ws.add_table(table)
    for col_idx, header in enumerate(df.columns, start=1):
        max_len = max([len(str(header))] + [len(safe_text(v)) for v in df[header].head(100)])
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len * 0.7, 11), 55)

# Abas dinâmicas.
write_dataframe_sheet(wb, 'Artefatos_Relevantes', ARTIFACTS, 'T_Artefatos_Relevantes')
write_dataframe_sheet(wb, 'Numeros_Evidencias', METRICS, 'T_Numeros_Evidencias')
write_dataframe_sheet(wb, 'FULL_por_Celula', FULL_BY_CELL, 'T_FULL_por_Celula')
write_dataframe_sheet(wb, 'Claims_AUD02', CLAIMS_VIEW, 'T_Claims_AUD02')
write_dataframe_sheet(wb, 'Crosscheck_AUD02', NUMERIC_CROSSCHECK, 'T_Crosscheck_AUD02')
write_dataframe_sheet(wb, 'Protocolos', PROTOCOLS, 'T_Protocolos')
write_dataframe_sheet(wb, 'Validacao_Geracao', VALIDATION, 'T_Validacao_Geracao')
add_status_formatting(wb['Validacao_Geracao'], 'status')
add_status_formatting(wb['Claims_AUD02'], 'status')
add_status_formatting(wb['Crosscheck_AUD02'], 'status')
add_status_formatting(wb['Artefatos_Relevantes'], 'validation_level')

# README reconstruído.
remove_sheet_if_exists(wb, 'README_Dominios')
readme = wb.create_sheet('README_Dominios', 0)
readme.merge_cells('A1:H1')
readme['A1'] = 'Matriz de Notebooks da Dissertação — versão 7.0'
readme['A1'].fill = TITLE_FILL
readme['A1'].font = TITLE_FONT
readme['A1'].alignment = Alignment(horizontal='center')
readme_rows = [
    ('Finalidade', 'Consolidar, navegar e conferir rapidamente notebooks, artefatos, números, protocolos, decisões e limitações.'),
    ('Fonte primária', 'Notebooks e artefatos oficiais. A matriz nunca substitui a fonte.'),
    ('AUD02', 'Auditoria transversal das claims definidas; linhas AUD02_AUDITED foram reproduzidas e verificadas.'),
    ('DOC01', 'Gera a matriz; não treina, não refaz splits e não altera o ramo experimental.'),
    ('Conteúdo curado', 'Indice_Notebooks, Metodo_Ciencia, Escrita_Dissertacao, Defesa_Validade e Acoes_Revisao.'),
    ('Conteúdo gerado', 'Artefatos_Relevantes, Numeros_Evidencias, FULL_por_Celula, Claims_AUD02, Crosscheck_AUD02, Protocolos e Validacao_Geracao.'),
    ('Níveis de validação', 'AUD02_AUDITED = coberto pela auditoria; ARTIFACT_INDEXED = localizado no inventário; PLANNED_NOT_GENERATED e ARCHIVED_ELSEWHERE = ausência classificada; UNRESOLVED_REFERENCE = requer revisão.'),
    ('Protocolos', 'TSCV para seleção/discriminação; teste fixo herdado para lift oficial e métricas operacionais.'),
    ('Regra de uso', 'Para citar um número, conferir artifact_path, field, selector, protocol, validation_level e hash.'),
    ('Gerado em', RUN_AT.isoformat()),
    ('Matriz-base', str(SOURCE_MATRIX_PATH)),
    ('AUD02', str(AUD02_DIR)),
    ('Inventário de 04-reports', str(REPORTS_INVENTORY_PATH)),
    ('Status geral', OVERALL_STATUS),
]
readme.append([])
for key, value in readme_rows:
    readme.append([key, value])
for row in readme.iter_rows(min_row=3, max_col=2):
    row[0].font = Font(bold=True, color='1F4E78')
    row[0].fill = SUBTITLE_FILL
    row[0].alignment = Alignment(vertical='top', wrap_text=True)
    row[1].alignment = Alignment(vertical='top', wrap_text=True)
    row[0].border = row[1].border = CELL_BORDER
readme.column_dimensions['A'].width = 24
readme.column_dimensions['B'].width = 105
readme.sheet_view.showGridLines = False

# Dashboard reconstruído.
remove_sheet_if_exists(wb, 'Dashboard')
dashboard = wb.create_sheet('Dashboard', 0)
dashboard.merge_cells('A1:F1')
dashboard['A1'] = 'Dashboard — Matriz Notebooks Dissertação v7.0'
dashboard['A1'].fill = TITLE_FILL
dashboard['A1'].font = TITLE_FONT
dashboard['A1'].alignment = Alignment(horizontal='center')

summary_rows = [
    ('Gerado em', RUN_AT.strftime('%Y-%m-%d %H:%M UTC')),
    ('Notebooks/documentos indexados', int(STATIC_SHEETS['Indice_Notebooks']['notebook_id'].nunique())),
    ('Artefatos relevantes', len(ARTIFACTS)),
    ('Artefatos AUD02_AUDITED', int((ARTIFACTS['validation_level'] == 'AUD02_AUDITED').sum())),
    ('Artefatos apenas indexados', int((ARTIFACTS['validation_level'] == 'ARTIFACT_INDEXED').sum())),
    ('Referências não resolvidas', int((ARTIFACTS['validation_level'] == 'UNRESOLVED_REFERENCE').sum())),
    ('Números/evidências atuais', len(METRICS)),
    ('Claims AUD02 confirmadas', int((AUD02_CLAIMS['status'] == 'CONFIRMED').sum())),
    ('Checks AUD02 PASS', int((AUD02_CHECKS['status'] == 'PASS').sum())),
    ('Células modeláveis', len(AUD02_SUMMARY['headline_results']['modelable_cells'])),
    ('ROC-AUC TSCV soberano', f"{headline['roc_auc_min']:.3f}–{headline['roc_auc_max']:.3f}"),
    ('Lift oficial teste fixo', f"{headline['lift_min']:.3f}–{headline['lift_max']:.3f}×"),
    ('Antecipação mediana', f"{100*headline['anticipation_median']:.1f}%"),
    ('LSTM promovida', 'nenhuma'),
    ('Status DOC01', OVERALL_STATUS),
]

dashboard.append([])
dashboard.append(['Indicador', 'Valor'])
for item in summary_rows:
    dashboard.append(list(item))
for cell in dashboard[3]:
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal='center')
for row in dashboard.iter_rows(min_row=4, max_col=2):
    row[0].font = Font(bold=True, color='1F4E78')
    row[0].fill = SUBTITLE_FILL
    for cell in row:
        cell.border = CELL_BORDER
        cell.alignment = Alignment(vertical='top', wrap_text=True)
dashboard.column_dimensions['A'].width = 34
dashboard.column_dimensions['B'].width = 45
dashboard.column_dimensions['D'].width = 24
dashboard.column_dimensions['E'].width = 14
dashboard.sheet_view.showGridLines = False

# Pequeno gráfico de cobertura de artefatos.
chart_start = 4
coverage = [
    ('AUD02_AUDITED', int((ARTIFACTS['validation_level'] == 'AUD02_AUDITED').sum())),
    ('ARTIFACT_INDEXED', int((ARTIFACTS['validation_level'] == 'ARTIFACT_INDEXED').sum())),
    ('PLANNED', int((ARTIFACTS['validation_level'] == 'PLANNED_NOT_GENERATED').sum())),
    ('ARCHIVED', int((ARTIFACTS['validation_level'] == 'ARCHIVED_ELSEWHERE').sum())),
    ('UNRESOLVED', int((ARTIFACTS['validation_level'] == 'UNRESOLVED_REFERENCE').sum())),
]
dashboard['D3'] = 'Cobertura'
dashboard['E3'] = 'Quantidade'
for c in dashboard['D3:E3'][0]:
    c.fill = HEADER_FILL
    c.font = HEADER_FONT
for i, (label, value) in enumerate(coverage, start=4):
    dashboard.cell(i, 4, label)
    dashboard.cell(i, 5, value)
bar = BarChart()
bar.type = 'col'
bar.style = 10
bar.title = 'Cobertura dos artefatos'
bar.legend = None
bar.y_axis.title = 'Quantidade'
bar.x_axis.title = 'Nível'
bar.add_data(Reference(dashboard, min_col=5, min_row=3, max_row=8), titles_from_data=True)
bar.set_categories(Reference(dashboard, min_col=4, min_row=4, max_row=8))
bar.height = 7
bar.width = 11
dashboard.add_chart(bar, 'D8')

# Lista de domínios: preservar ou criar simplificada e ocultar.
if 'Listas_Dominio' not in wb.sheetnames:
    list_ws = wb.create_sheet('Listas_Dominio')
    list_ws.append(['validation_level', 'status', 'pipeline'])
    values = sorted(set(ARTIFACTS['validation_level']))
    for i, value in enumerate(values, start=2):
        list_ws.cell(i, 1, value)
else:
    list_ws = wb['Listas_Dominio']
list_ws.sheet_state = 'hidden'

# Ordem final.
order = [
    'Dashboard', 'README_Dominios', 'Indice_Notebooks', 'Metodo_Ciencia',
    'Escrita_Dissertacao', 'Defesa_Validade', 'Acoes_Revisao',
    'Artefatos_Relevantes', 'Numeros_Evidencias', 'FULL_por_Celula',
    'Claims_AUD02', 'Crosscheck_AUD02', 'Protocolos', 'Validacao_Geracao', 'Listas_Dominio'
]
existing = {ws.title: ws for ws in wb.worksheets}
wb._sheets = [existing[name] for name in order if name in existing]

# Propriedades.
wb.properties.title = 'Matriz de Notebooks da Dissertação v7.0'
wb.properties.subject = 'Consolidação documental e rastreabilidade'
wb.properties.creator = 'Sérgio Henrique Cerqueira Costa / DOC01'
wb.properties.description = 'Gerada a partir da Matriz v5, AUD02 v1.2 e snapshot runtime do 04-reports vigente.'
wb.calculation.fullCalcOnLoad = True
wb.calculation.forceFullCalc = True

wb.save(OUTPUT_MATRIX_PATH)
print('Matriz gerada:', OUTPUT_MATRIX_PATH)
print('Tamanho:', OUTPUT_MATRIX_PATH.stat().st_size, 'bytes')

# ============================================================
# 8. Exportação dos registros, resumo e manifesto DOC01
# ============================================================

ARTIFACT_REGISTRY_PATH = OUTPUT_DIR / 'DOC01_artifact_registry.csv'
METRIC_REGISTRY_PATH = OUTPUT_DIR / 'DOC01_metric_registry.csv'
VALIDATION_PATH = OUTPUT_DIR / 'DOC01_generation_validation.csv'
CROSSCHECK_PATH = OUTPUT_DIR / 'DOC01_numeric_crosscheck.csv'
SUMMARY_PATH = OUTPUT_DIR / 'DOC01_summary.json'
MANIFEST_PATH = OUTPUT_DIR / 'DOC01_manifest_sha256.csv'

ARTIFACTS.to_csv(ARTIFACT_REGISTRY_PATH, index=False, encoding='utf-8')
METRICS.to_csv(METRIC_REGISTRY_PATH, index=False, encoding='utf-8')
VALIDATION.to_csv(VALIDATION_PATH, index=False, encoding='utf-8')
NUMERIC_CROSSCHECK.to_csv(CROSSCHECK_PATH, index=False, encoding='utf-8')

summary = {
    'notebook': 'DOC01_geracao_matriz_notebooks.ipynb',
    'doc01_version': DOC01_VERSION,
    'matrix_version': MATRIX_VERSION,
    'generated_at_utc': RUN_AT.isoformat(),
    'role': 'documentacao_consolidacao_somente_leitura',
    'source_matrix': str(SOURCE_MATRIX_PATH),
    'aud02_directory': str(AUD02_DIR),
    'reports_inventory': str(REPORTS_INVENTORY_PATH),
    'output_matrix': str(OUTPUT_MATRIX_PATH),
    'source_hierarchy': [
        'notebooks_and_official_artifacts',
        'AUD02_transversal_audit',
        'DOC01_matrix_consolidation',
        'writing_map',
    ],
    'counts': {
        'static_sheets': len(STATIC_SHEETS),
        'indexed_notebooks_documents': int(STATIC_SHEETS['Indice_Notebooks']['notebook_id'].nunique()),
        'artifacts': int(len(ARTIFACTS)),
        'aud02_audited_artifacts': int((ARTIFACTS['validation_level'] == 'AUD02_AUDITED').sum()),
        'artifact_indexed': int((ARTIFACTS['validation_level'] == 'ARTIFACT_INDEXED').sum()),
        'unresolved_references': int((ARTIFACTS['validation_level'] == 'UNRESOLVED_REFERENCE').sum()),
        'planned_not_generated': int((ARTIFACTS['validation_level'] == 'PLANNED_NOT_GENERATED').sum()),
        'archived_elsewhere': int((ARTIFACTS['validation_level'] == 'ARCHIVED_ELSEWHERE').sum()),
        'low_confidence_resolutions': int((ARTIFACTS['resolution_confidence'] == 'LOW').sum()),
        'metrics': int(len(METRICS)),
        'cells': int(len(FULL_BY_CELL)),
        'claims': int(len(CLAIMS_VIEW)),
        'validation_pass': int((VALIDATION['status'] == 'PASS').sum()),
        'validation_review': int((VALIDATION['status'] == 'REVIEW').sum()),
        'validation_fail': int((VALIDATION['status'] == 'FAIL').sum()),
    },
    'overall_status': OVERALL_STATUS,
    'interpretation': (
        'PASS confirma geração sem falhas nem revisões pendentes; PASS_WITH_REVIEW indica apenas pendências documentais não bloqueantes. '
        'Linhas ARTIFACT_INDEXED não equivalem a claims auditadas pelo AUD02.'
    ),
    'regeneration_rule': (
        'Reexecutar DOC01 quando houver mudança em artefatos oficiais, na Matriz v5 curada '
        'ou nas regras declarativas do próprio DOC01.'
    ),
}
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Manifesto sem autorreferência.
manifest_rows = []
for path in [
    OUTPUT_MATRIX_PATH,
    ARTIFACT_REGISTRY_PATH,
    METRIC_REGISTRY_PATH,
    VALIDATION_PATH,
    CROSSCHECK_PATH,
    SUMMARY_PATH,
]:
    digest, hash_status = sha256_file(path)
    manifest_rows.append({
        'filename': path.name,
        'path': str(path),
        'size_bytes': path.stat().st_size,
        'sha256': digest or '',
        'hash_status': hash_status,
    })
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_PATH, index=False, encoding='utf-8')

# Verificação de reabertura do XLSX.
check_wb = load_workbook(OUTPUT_MATRIX_PATH, read_only=True, data_only=False)
expected_sheets = [
    'Dashboard', 'README_Dominios', 'Indice_Notebooks', 'Metodo_Ciencia',
    'Escrita_Dissertacao', 'Defesa_Validade', 'Acoes_Revisao',
    'Artefatos_Relevantes', 'Numeros_Evidencias', 'FULL_por_Celula',
    'Claims_AUD02', 'Crosscheck_AUD02', 'Protocolos', 'Validacao_Geracao', 'Listas_Dominio'
]
missing_sheets = [name for name in expected_sheets if name not in check_wb.sheetnames]
if missing_sheets:
    raise RuntimeError(f'Planilha final sem abas obrigatórias: {missing_sheets}')

print('\nDOC01 concluído.')
print('Status:', summary['overall_status'])
print('Matriz:', OUTPUT_MATRIX_PATH)
print('Artefatos:', ARTIFACT_REGISTRY_PATH)
print('Métricas:', METRIC_REGISTRY_PATH)
print('Validação:', VALIDATION_PATH)
print('Cross-check:', CROSSCHECK_PATH)
print('Resumo:', SUMMARY_PATH)
print('Manifesto:', MANIFEST_PATH)
print('Abas:', check_wb.sheetnames)

display(pd.DataFrame([summary['counts']]))

Mounted at /content/drive
DOC01_VERSION        : 1.5
MATRIX_VERSION       : 7.0
IN_COLAB             : True
PROJECT_ROOT         : /content/drive/MyDrive/Mestrado
SOURCE_MATRIX_PATH   : /content/drive/MyDrive/Mestrado/Matriz_Notebooks_Dissertacao_v5.xlsx
SEARCH_ROOTS         : ['/content/drive/MyDrive', '/content/drive/Shareddrives']
AUD02_DIR            : /content/drive/MyDrive/Mestrado/04-reports/AUD02_evidence_audit
REPORTS_INVENTORY    : /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/04-reports_inventory_runtime.csv
INVENTORY_MODE       : RUNTIME_CURRENT_04_REPORTS
OUTPUT_DIR           : /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation
AUD02 checks: {'PASS': 18, 'REVIEW': 1}
Static sheets loaded: {'Indice_Notebooks': 32, 'Metodo_Ciencia': 32, 'Escrita_Dissertacao': 32, 'Defesa_Validade': 32, 'Acoes_Revisao': 33}
Curadoria aplicada.
Artefatos registrados: 267
Níveis: {'ARTIFACT_INDEXED': 186, 'AUD02_AUDITED': 48, 'REFERENCED_OUTSIDE_REPORT

,check_id,description,observed,expected,status,action
0,D01,Matriz v5 localizada.,/content/drive/MyDrive/Mestrado/Matriz_Noteboo...,arquivo existente,PASS,
1,D02,AUD02 v1.2 sem falha material; REVIEW de regre...,PASS_WITH_REGRESSION_REVIEW,status não iniciado por FAIL,PASS,
2,D03,Todos os checks do AUD02 aprovados sem falha m...,"{'PASS': 18, 'REVIEW': 1}",nenhum FAIL,PASS,
3,D04,FULL por célula contém a–h.,"['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']","['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']",PASS,
4,D05,Única célula não modelável é d.,['d'],['d'],PASS,
5,D06,Lift oficial usa protocolo alinhado.,['ALIGNED_NB11_FIXED_INHERITED_TEST'],['ALIGNED_NB11_FIXED_INHERITED_TEST'],PASS,
6,D07,Nenhuma LSTM promovida.,[],[],PASS,
7,D08,Artefatos relevantes indexados.,267,> 0,PASS,
8,D09,Referências realmente não resolvidas após alia...,0,0,PASS,Apenas UNRESOLVED_REFERENCE exige inspeção; pl...
9,D10,Hashes divergentes entre fontes auditadas.,0,0,PASS,


Matriz gerada: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/Matriz_Notebooks_Dissertacao_v7_0.xlsx
Tamanho: 199755 bytes

DOC01 concluído.
Status: PASS
Matriz: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/Matriz_Notebooks_Dissertacao_v7_0.xlsx
Artefatos: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_artifact_registry.csv
Métricas: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_metric_registry.csv
Validação: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_generation_validation.csv
Cross-check: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_numeric_crosscheck.csv
Resumo: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_summary.json
Manifesto: /content/drive/MyDrive/Mestrado/04-reports/DOC01_matrix_consolidation/DOC01_manifest_sha256.csv
Abas: ['Dashboard', 'README_Dominios', 'Indice_Notebooks', 'Me

,static_sheets,indexed_notebooks_documents,artifacts,aud02_audited_artifacts,artifact_indexed,unresolved_references,planned_not_generated,archived_elsewhere,low_confidence_resolutions,metrics,cells,claims,validation_pass,validation_review,validation_fail
0,5,34,267,48,186,0,3,2,0,142,8,9,18,0,0


## 1. Resultado da consolidação documental

O DOC01 concluiu a geração da **Matriz de Notebooks da Dissertação v7.0** com status geral `PASS`. A etapa permaneceu exclusivamente documental e de rastreabilidade: não treinou modelos, não refez divisões temporais e não alterou os artefatos experimentais. A consolidação utilizou a Matriz v5 como base semântica curada, o AUD02 como auditoria da linhagem vigente e um inventário runtime do `04-reports`.

## 2. Validação da geração

Os **18 checks D01–D18** foram aprovados, sem `REVIEW` ou `FAIL` no DOC01. A validação confirmou, entre outros pontos, a localização da Matriz v5, a ausência de falha material no AUD02, a presença das oito células, a célula `d` como única não modelável, o protocolo oficial do lift no teste fixo herdado, a ausência de promoção da LSTM, a inexistência de referências não resolvidas, a ausência de divergências de hash e a remoção das formulações críticas antigas da curadoria ativa.

O cross-check contra o AUD02 totalizou **53 verificações**, todas com `PASS`. Foram reproduzidos os indicadores de síntese e os valores por célula, incluindo a faixa de ROC-AUC soberano de **0,636979 a 0,812583**, o lift oficial de **1,545640× a 5,771561×**, a mediana de antecipação de **33,33%**, as sete células modeláveis (`a`, `b`, `c`, `e`, `f`, `g`, `h`), seis células acima da referência e somente `b` no gate nominal da LSTM, sem promoção como fonte primária.

## 3. Rastreabilidade e cobertura dos artefatos

A matriz registra **267 artefatos**: 48 classificados como `AUD02_AUDITED`, 186 como `ARTIFACT_INDEXED`, 28 como `REFERENCED_OUTSIDE_REPORTS`, 3 como `PLANNED_NOT_GENERATED` e 2 como `ARCHIVED_ELSEWHERE`. Não restaram referências `UNRESOLVED_REFERENCE` nem resoluções com confiança `LOW`.

Foram consolidadas **142 métricas**, todas com fonte, campo e nível de validação preenchidos. A cobertura documental inclui 34 notebooks/documentos indexados, 8 células, 9 claims e 6 protocolos.

## 4. Integridade dos produtos do DOC01

O manifesto do DOC01 registra seis produtos governantes: a Matriz v7.0, os registros de artefatos e métricas, a validação da geração, o cross-check numérico e o resumo da etapa. Na validação do Grupo 3, os seis arquivos foram conferidos novamente e seus tamanhos e SHA-256 coincidiram integralmente com o manifesto.

A Matriz v7.0 contém **15 abas**: `Dashboard`, `README_Dominios`, `Indice_Notebooks`, `Metodo_Ciencia`, `Escrita_Dissertacao`, `Defesa_Validade`, `Acoes_Revisao`, `Artefatos_Relevantes`, `Numeros_Evidencias`, `FULL_por_Celula`, `Claims_AUD02`, `Crosscheck_AUD02`, `Protocolos`, `Validacao_Geracao` e `Listas_Dominio`.

A identidade documental ficou uniforme como **v7.0** no nome do arquivo, no Dashboard, no README e nas propriedades do workbook. Não foram encontradas ocorrências residuais de `v7.1` nem erros de fórmula aparentes (`#REF!`, `#DIV/0!`, `#VALUE!`, `#NAME?` ou `#N/A`).

## 5. Relação com o AUD02 e com a linhagem anterior

O `REVIEW` R10 do AUD02 permanece uma observação de regressão não bloqueante associada à exclusão da célula `d` da comparação soberana por não modelabilidade. Ele não se converteu em pendência do DOC01: a consolidação foi executada com **18 validações próprias em `PASS` e nenhuma revisão pendente**.

A Matriz v7.0 passa, assim, a representar documentalmente a linhagem corrigida e auditada pelo AUD02. A Matriz v6.1 permanece como registro histórico da linhagem anterior e não deve ser utilizada como fonte corrente para novos números ou decisões documentais.

## 6. Fechamento

O DOC01 está **aprovado no Grupo 3**. A Matriz de Notebooks da Dissertação v7.0 está apta a assumir o papel de matriz documental vigente para a próxima etapa de atualização dos instrumentos de escrita e de figuras, preservando os artefatos experimentais como fontes primárias das afirmações científicas.
